In [23]:
from lunar_python import Solar, Lunar

# 1. Initialize with a Solar Date (Year, Month, Day, Hour, Minute, Second)
# Example: May 15, 2000, at 12:00:00
solar = Solar.fromYmdHms(1985, 11, 25, 16, 15, 0)

# 2. Convert to Lunar object (which holds the Bazi data)
lunar = solar.getLunar()

# 3. Get the "Eight Char" (BaZi) object
bazi = lunar.getEightChar()

# 4. Display the results
print(f"Year Pillar:  {bazi.getYear()} ({bazi.getYearWuXing()} - {bazi.getYearZhi()})")
print(f"Month Pillar: {bazi.getMonth()} ({bazi.getMonthWuXing()})")
print(f"Day Pillar:   {bazi.getDay()} ({bazi.getDayWuXing()})")
print(f"Hour Pillar:  {bazi.getTime()} ({bazi.getTimeWuXing()})")

# 5. Access specific Stems (Gan) and Branches (Zhi)
print(f"Day Master (Heavenly Stem of the Day): {bazi.getDayGan()}")

Year Pillar:  乙丑 (木土 - 丑)
Month Pillar: 丁亥 (火水)
Day Pillar:   戊辰 (土土)
Hour Pillar:  庚申 (金金)
Day Master (Heavenly Stem of the Day): 戊


In [ ]:
# Corinne's birthday example
from datetime import datetime
from src.astronomer_calculations.true_solar_time import get_true_solar_time

solar_birthday= Solar.fromYmdHms(1987, 6, 3, 12, 6, 0)  # Create solar date June 3, 1987 at 12:06 PM
tst_birthday, inputs_report = get_true_solar_time(datetime(1987, 6, 3, 12, 6, 0), 1.4759, 103.808053)  # Get true solar time for the birthday


In [1]:
from lunar_python import Solar
import math
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo
from datetime import datetime

def get_true_solar_time(datetime, latitude, longitude):
    """
    Convert standard clock time to true solar time.

    Args:
        dt: datetime object with standard clock time
        longitude: Observer's longitude in decimal degrees
                   (positive for East, negative for West)

    Returns:
        tuple: (Solar object with true solar time, dict with calculation details)
    """
    year = datetime.year
    month = datetime.month
    day = datetime.day
    hour = datetime.hour
    minute = datetime.minute
    second = datetime.second
    longitude = longitude
    latitude = latitude

    # 1. Use TimezoneFinder to get the NAME of the timezone
    tf = TimezoneFinder()
    tz_name = tf.timezone_at(lat=latitude, lng=longitude) # "Asia/Singapore"

    # 2. Use ZoneInfo to find the UTC offset for that specific DATE
    tz = ZoneInfo(tz_name)

    # The .utcoffset() method handles DST and historical shifts automatically
    offset_delta = tz.utcoffset(datetime)
    utc_offset = offset_delta.total_seconds() / 3600

    # 3. Calculate Standard Meridian
    standard_meridian = utc_offset * 15

    # Step 1: Calculate day of year (n)
    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

    # Check for leap year
    if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
        days_in_month[1] = 29

    n = sum(days_in_month[:month-1]) + day

    # Step 2: Calculate Equation of Time (EoT)
    B = (360 / 365) * (n - 81)  # in degrees
    B_rad = math.radians(B)

    eot_minutes = 9.87 * math.sin(2 * B_rad) - 7.53 * math.cos(B_rad) - 1.5 * math.sin(B_rad)

    # Step 3: Calculate Longitude Correction
    longitude_correction = 4 * (longitude - standard_meridian)  # in minutes

    # Step 4: Apply total adjustment
    total_adjustment_minutes = longitude_correction + eot_minutes

    # Convert current time to total seconds
    total_seconds = hour * 3600 + minute * 60 + second
    total_seconds += int(total_adjustment_minutes * 60)

    # Handle day overflow
    day_offset = total_seconds // 86400
    total_seconds = total_seconds % 86400

    adjusted_hour = total_seconds // 3600
    adjusted_minute = (total_seconds % 3600) // 60
    adjusted_second = total_seconds % 60

    # Create new Solar object with adjusted time
    adjusted_solar = Solar.fromYmdHms(
        year, month, day + day_offset,
        adjusted_hour, adjusted_minute, adjusted_second
    )

    return adjusted_solar, {
        'datetime': dt,
        'latitude': latitude,
        'longitude': longitude,
        'standard_meridian': standard_meridian,
        'day_of_year': n,
        'eot_minutes': eot_minutes,
        'longitude_correction': longitude_correction,
        'total_adjustment_minutes': total_adjustment_minutes
    }

# Usage
dt = datetime(1985, 11, 25, 17, 7, 0)
tst, details = get_true_solar_time(dt, 1.351616, 103.808053)  # Singapore coordinates
# dt = datetime(1987, 6, 3, 12, 6, 0)
# tst, details = get_true_solar_time(dt, 1.4759, 103.808053)


print(f"Standard Clock Time: {details['datetime']}")
print(f"Latitude: {details['latitude']:.4f}°N")
print(f"Longitude: {details['longitude']:.4f}°E")
print(f"Standard Meridian: {details['standard_meridian']}°")
print(f"Day of Year: {details['day_of_year']}")
print(f"Equation of Time: {details['eot_minutes']:.2f} minutes")
print(f"Longitude Correction: {details['longitude_correction']:.2f} minutes")
print(f"Total Adjustment: {details['total_adjustment_minutes']:.2f} minutes")
print(f"\nTrue Solar Time: {tst.toFullString()}")

# Get True Solar Time BaZi
lunar = tst.getLunar()
bazi_tst = lunar.getEightChar()
print(f"\nBaZi: {bazi_tst.getYear()}, {bazi_tst.getMonth()}, {bazi_tst.getDay()}, {bazi_tst.getTime()}")

# Get Standard Clock Time BaZi
solar = Solar.fromYmdHms(1987, 6, 3, 12, 6, 0)
lunar_std = solar.getLunar()
bazi_std = lunar_std.getEightChar()
print(f"\nBaZi: {bazi_std.getYear()}, {bazi_std.getMonth()}, {bazi_std.getDay()}, {bazi_std.getTime()}")

Standard Clock Time: 1985-11-25 17:07:00
Latitude: 1.3516°N
Longitude: 103.8081°E
Standard Meridian: 120.0°
Day of Year: 329
Equation of Time: 12.23 minutes
Longitude Correction: -64.77 minutes
Total Adjustment: -52.54 minutes

True Solar Time: 1985-11-25 16:14:28 星期一 射手座

BaZi: 乙丑, 丁亥, 戊辰, 庚申

BaZi: 丁卯, 乙巳, 癸未, 戊午


In [25]:
def get_bazi_elements(bazi):
    """
    Analyzes the Five Elements (Wu Xing) from a BaZi object.

    Returns:
        dict: A structured report of elements in the 4 pillars.
    """

    # 1. Define the components
    pillars = {
        "Year":  {"stem": bazi.getYearGan(),  "branch": bazi.getYearZhi()},
        "Month": {"stem": bazi.getMonthGan(), "branch": bazi.getMonthZhi()},
        "Day":   {"stem": bazi.getDayGan(),   "branch": bazi.getDayZhi()},
        "Hour":  {"stem": bazi.getTimeGan(),  "branch": bazi.getTimeZhi()}
    }

    element_report = {}
    total_counts = {"Wood": 0, "Fire": 0, "Earth": 0, "Metal": 0, "Water": 0}

    for name, p in pillars.items():
        # Get Element of the Stem (Top half)
        stem_element = bazi.getYearWuXing()[0] # This is a bit of a shortcut
        # More robust: use the specific pillar's WuXing
        if name == "Year": ux = bazi.getYearWuXing()
        elif name == "Month": ux = bazi.getMonthWuXing()
        elif name == "Day": ux = bazi.getDayWuXing()
        else: ux = bazi.getTimeWuXing()

        # ux is a string like "木土" (Wood Earth) where [0] is Stem, [1] is Branch
        element_report[name] = {
            "Stem": p['stem'],
            "Stem_Element": ux[0],
            "Branch": p['branch'],
            "Branch_Element": ux[1],
            "Hidden_Stems": bazi.getYearGan() # This logic needs manual mapping or specific methods
        }

        # Increment totals
        total_counts[translate_element(ux[0])] += 1
        total_counts[translate_element(ux[1])] += 1

    return element_report, total_counts

def translate_element(chinese_char):
    mapping = {"木": "Wood", "火": "Fire", "土": "Earth", "金": "Metal", "水": "Water"}
    return mapping.get(chinese_char, "Unknown")

# Get BaZi Five Elements Balance. Use tst Bazi
report_tst, totals_tst = get_bazi_elements(bazi_tst)
print("\n--- Five Elements Balance (True Solar Time) ---")
for el, count in totals_tst.items():
    print(f"{el}: {'I' * count} ({count})")

# Get BaZi Five Elements Balance. Use std Bazi
report_std, totals_std = get_bazi_elements(bazi_std)
print("\n--- Five Elements Balance (Standard Time) ---")
for el, count in totals_std.items():
    print(f"{el}: {'I' * count} ({count})")


--- Five Elements Balance (True Solar Time) ---
Wood: II (2)
Fire: III (3)
Earth: II (2)
Metal:  (0)
Water: I (1)

--- Five Elements Balance (Standard Time) ---
Wood: II (2)
Fire: III (3)
Earth: II (2)
Metal:  (0)
Water: I (1)


In [26]:
def analyze_bazi_strength(bazi):
    translate = {
        "甲": "Wood", "乙": "Wood", "丙": "Fire", "丁": "Fire",
        "戊": "Earth", "己": "Earth", "庚": "Metal", "辛": "Metal",
        "壬": "Water", "癸": "Water"
    }

    # 1. Day Master (The Reference Point)
    dm_gan = bazi.getDayGan()
    dm_el = translate.get(dm_gan)

    # Define Support: Self and Resource (Mother)
    cycle = ["Wood", "Fire", "Earth", "Metal", "Water"]
    resource_el = cycle[(cycle.index(dm_el) - 1) % 5]

    print(f"[STEP 1] Day Master is {dm_gan}({dm_el}).")
    print(f"         Supporting elements: {dm_el} (Self) and {resource_el} (Resource).")

    # 2. Professional Weights (Total = 10.0)
    # The Month Branch (Season) is ~30-40% of the total power.
    weights = {
        "Year_Stem": 1.0, "Year_Branch": 1.0,
        "Month_Stem": 1.0, "Month_Branch": 3.0, # The "Command"
        "Day_Stem": 0.0, # DM itself isn't counted against itself
        "Day_Branch": 2.0, # The "Root"
        "Hour_Stem": 1.0, "Hour_Branch": 1.0
    }

    scores = {"Wood": 0.0, "Fire": 0.0, "Earth": 0.0, "Metal": 0.0, "Water": 0.0}
    logs = []

    # 3. Processing Pillars
    pillar_map = [
        ("Year", bazi.getYearGan(), bazi.getYearZhi(), "Year_Stem", "Year_Branch"),
        ("Month", bazi.getMonthGan(), bazi.getMonthZhi(), "Month_Stem", "Month_Branch"),
        ("Day", bazi.getDayGan(), bazi.getDayZhi(), "Day_Stem", "Day_Branch"),
        ("Hour", bazi.getTimeGan(), bazi.getTimeZhi(), "Hour_Stem", "Hour_Branch")
    ]

    print("\n[STEP 2] Calculating Positional Weights...")
    for name, gan, zhi, s_key, b_key in pillar_map:
        # Score Stem
        s_el = translate.get(gan)
        s_weight = weights[s_key]
        if s_weight > 0:
            scores[s_el] += s_weight
            logs.append(f"{name} Stem ({gan}): +{s_weight} to {s_el}")

        # Score Branch (Main Element)
        # Note: Professional branches are complex; we use the main element (Zhi WuXing)
        # We find the branch element by looking at the first character of the WuXing string
        b_wx_str = ""
        if name == "Year": b_wx_str = bazi.getYearWuXing()
        elif name == "Month": b_wx_str = bazi.getMonthWuXing()
        elif name == "Day": b_wx_str = bazi.getDayWuXing()
        else: b_wx_str = bazi.getTimeWuXing()

        b_el = translate_wx_char(b_wx_str[1]) # [1] is the Branch's element
        b_weight = weights[b_key]
        scores[b_el] += b_weight
        logs.append(f"{name} Branch ({zhi}): +{b_weight} to {b_el}")

    for entry in logs: print(f"         {entry}")

    # 4. Final Calculation
    total_support = scores[dm_el] + scores[resource_el]
    # In a 10-point system, > 5.0 is Strong.
    is_strong = total_support >= 5.0

    print(f"\n[STEP 3] Final Totals:")
    print(f"         Self ({dm_el}): {scores[dm_el]:.1f}")
    print(f"         Resource ({resource_el}): {scores[resource_el]:.1f}")
    print(f"         Total Support Score: {total_support:.1f} / 10.0")

    return {
        "dm": f"{dm_gan}({dm_el})",
        "is_strong": is_strong,
        "scores": scores,
        "total_support": total_support
    }

def translate_wx_char(c):
    m = {"木": "Wood", "火": "Fire", "土": "Earth", "金": "Metal", "水": "Water"}
    return m.get(c, "Unknown")

# usage
print("\n--- BaZi Strength Analysis (True Solar Time) ---")
analysis_tst = analyze_bazi_strength(bazi_tst)

print("\n--- BaZi Strength Analysis (Standard Time) ---")
analysis_std = analyze_bazi_strength(bazi_std)


--- BaZi Strength Analysis (True Solar Time) ---
[STEP 1] Day Master is 癸(Water).
         Supporting elements: Water (Self) and Metal (Resource).

[STEP 2] Calculating Positional Weights...
         Year Stem (丁): +1.0 to Fire
         Year Branch (卯): +1.0 to Wood
         Month Stem (乙): +1.0 to Wood
         Month Branch (巳): +3.0 to Fire
         Day Branch (未): +2.0 to Earth
         Hour Stem (戊): +1.0 to Earth
         Hour Branch (午): +1.0 to Fire

[STEP 3] Final Totals:
         Self (Water): 0.0
         Resource (Metal): 0.0
         Total Support Score: 0.0 / 10.0

--- BaZi Strength Analysis (Standard Time) ---
[STEP 1] Day Master is 癸(Water).
         Supporting elements: Water (Self) and Metal (Resource).

[STEP 2] Calculating Positional Weights...
         Year Stem (丁): +1.0 to Fire
         Year Branch (卯): +1.0 to Wood
         Month Stem (乙): +1.0 to Wood
         Month Branch (巳): +3.0 to Fire
         Day Branch (未): +2.0 to Earth
         Hour Stem (戊): +1.0 to 

In [27]:
def compare_bazi_strength(bazi_tst, bazi_std):
    def get_stats(bazi):
        translate = {
            "甲": "Wood", "乙": "Wood", "丙": "Fire", "丁": "Fire",
            "戊": "Earth", "己": "Earth", "庚": "Metal", "辛": "Metal",
            "壬": "Water", "癸": "Water"
        }

        dm_gan = bazi.getDayGan()
        dm_el = translate.get(dm_gan)
        cycle = ["Wood", "Fire", "Earth", "Metal", "Water"]
        resource_el = cycle[(cycle.index(dm_el) - 1) % 5]

        weights = {
            "Year_Stem": 1.0, "Year_Branch": 1.0,
            "Month_Stem": 1.0, "Month_Branch": 3.0,
            "Day_Stem": 0.0, "Day_Branch": 2.0,
            "Hour_Stem": 1.0, "Hour_Branch": 1.0
        }

        scores = {"Wood": 0.0, "Fire": 0.0, "Earth": 0.0, "Metal": 0.0, "Water": 0.0}

        # Pillar Data
        p_data = [
            (bazi.getYearGan(), bazi.getYearZhi(), bazi.getYearWuXing(), "Year_Stem", "Year_Branch"),
            (bazi.getMonthGan(), bazi.getMonthZhi(), bazi.getMonthWuXing(), "Month_Stem", "Month_Branch"),
            (bazi.getDayGan(), bazi.getDayZhi(), bazi.getDayWuXing(), "Day_Stem", "Day_Branch"),
            (bazi.getTimeGan(), bazi.getTimeZhi(), bazi.getTimeWuXing(), "Hour_Stem", "Hour_Branch")
        ]

        for gan, zhi, wx, s_key, b_key in p_data:
            s_el = translate.get(gan)
            if weights[s_key] > 0: scores[s_el] += weights[s_key]
            b_el = translate_wx_char(wx[1])
            scores[b_el] += weights[b_key]

        total_support = scores[dm_el] + scores[resource_el]
        return {
            "dm": f"{dm_gan}({dm_el})",
            "pillars": f"{bazi.getYear()} {bazi.getMonth()} {bazi.getDay()} {bazi.getTime()}",
            "support": total_support,
            "status": "STRONG" if total_support >= 5.0 else "WEAK",
            "scores": scores
        }

    tst = get_stats(bazi_tst)
    std = get_stats(bazi_std)

    print(f"{'Metric':<20} | {'Standard Clock Time':<25} | {'True Solar Time':<25}")
    print("-" * 75)
    print(f"{'Four Pillars':<20} | {std['pillars']:<25} | {tst['pillars']:<25}")
    print(f"{'Day Master':<20} | {std['dm']:<25} | {tst['dm']:<25}")
    print(f"{'Support Score':<20} | {std['support']:<25.1f} | {tst['support']:<25.1f}")
    print(f"{'Strength':<20} | {std['status']:<25} | {tst['status']:<25}")
    print("-" * 75)
    print("Detailed Element Scores:")
    for el in ["Wood", "Fire", "Earth", "Metal", "Water"]:
        print(f"{el:<20} | {std['scores'][el]:<25.1f} | {tst['scores'][el]:<25.1f}")

# Usage
compare_bazi_strength(bazi_tst, bazi_std)

Metric               | Standard Clock Time       | True Solar Time          
---------------------------------------------------------------------------
Four Pillars         | 丁卯 乙巳 癸未 戊午               | 丁卯 乙巳 癸未 戊午              
Day Master           | 癸(Water)                  | 癸(Water)                 
Support Score        | 0.0                       | 0.0                      
Strength             | WEAK                      | WEAK                     
---------------------------------------------------------------------------
Detailed Element Scores:
Wood                 | 2.0                       | 2.0                      
Fire                 | 5.0                       | 5.0                      
Earth                | 3.0                       | 3.0                      
Metal                | 0.0                       | 0.0                      
Water                | 0.0                       | 0.0                      


In [28]:
def analyze_ten_gods_native(bazi):
    # Standard labels for the output
    pillars = ["Year", "Month", "Day", "Hour"]

    # 1. Get the Ten Gods for the Stems (The visible traits)
    # Note: Day Stem is always 'Self' (比肩/Day Master), so we often skip or label it.
    stem_gods = [
        bazi.getYearShiShenGan(),
        bazi.getMonthShiShenGan(),
        "Day Master", # The Day Stem is you
        bazi.getTimeShiShenGan()
    ]

    # 2. Get the Ten Gods for the Hidden Stems (The internal potential)
    # These return lists of strings
    hidden_gods = [
        bazi.getYearShiShenZhi(),
        bazi.getMonthShiShenZhi(),
        bazi.getDayShiShenZhi(),
        bazi.getTimeShiShenZhi()
    ]

    print(f"{'Pillar':<10} | {'Stem God (Visible)':<20} | {'Hidden Gods (Internal)'}")
    print("-" * 65)

    for i in range(4):
        hidden_str = ", ".join(hidden_gods[i])
        print(f"{pillars[i]:<10} | {stem_gods[i]:<20} | {hidden_str}")

# Usage
print(f"Ten Gods Analysis for {bazi_tst.getDayGan()} Day Master:")
analyze_ten_gods_native(bazi_tst)
print("\n")
print(f"Ten Gods Analysis for {bazi_std.getDayGan()} Day Master:")
analyze_ten_gods_native(bazi_std)

Ten Gods Analysis for 癸 Day Master:
Pillar     | Stem God (Visible)   | Hidden Gods (Internal)
-----------------------------------------------------------------
Year       | 偏财                   | 食神
Month      | 食神                   | 正财, 正印, 正官
Day        | Day Master           | 七杀, 偏财, 食神
Hour       | 正官                   | 偏财, 七杀


Ten Gods Analysis for 癸 Day Master:
Pillar     | Stem God (Visible)   | Hidden Gods (Internal)
-----------------------------------------------------------------
Year       | 偏财                   | 食神
Month      | 食神                   | 正财, 正印, 正官
Day        | Day Master           | 七杀, 偏财, 食神
Hour       | 正官                   | 偏财, 七杀


3. How to Interpret Your Ten Gods
This is how you translate the code output into the life categories you asked for:

💰 Money (Wealth Elements)
Direct Wealth (正财): Stable income, salary, hard-earned money.

Indirect Wealth (偏财): Windfalls, investments, business ventures, "fast" money.

💼 Career & Authority (Officer Elements)
Direct Officer (正官): Respect for law, traditional career, management, boss.

Seven Killings (七杀): Ambition, risk-taking, entrepreneurship, pressure, authority through power.

❤️ Love & Marriage
For Men: Wealth elements (财) represent the wife/girlfriend.

For Women: Officer elements (官/杀) represent the husband/boyfriend.

Day Branch: The "Spouse Palace." The Ten God sitting in your Day Branch (hidden) shows the personality of your partner.

👨‍👩‍👧 Parents & Children
Resources (印): Represent parents and mentors (those who protect you).

Output (食/伤): Represent children (what you produce/nurture).

What these "Gods" mean for your Life
Once you run the code, you'll see Chinese terms. Here is how to map them to your specific questions:

💰 Money & Wealth
正财 (Direct Wealth): Regular income. Good for stability and long-term accumulation.

偏财 (Indirect Wealth): Side hustles, big investments, or unexpected windfalls.

💼 Career & Authority
正官 (Direct Officer): Corporate ladder, administrative roles, being a "proper" leader.

七杀 (Seven Killings): Disruption, high-pressure leadership, military/police, or entrepreneurship.

🎨 Intelligence & Talent
食神 (Eating God): Refined taste, deep expertise, calm creativity.

伤官 (Hurting Officer): Innovation, showmanship, aggressive intelligence, "thinking outside the box."

📚 Support & Health
正印 (Direct Resource): Academic success, mentors, traditional medicine.

偏印 (Indirect Resource): Self-taught skills, unconventional wisdom, intuition.

Why we check both "Stem" and "Hidden"
Stem God: This is how you behave in public. If your Year Stem is Direct Officer, people think you are very disciplined.

Hidden Gods: This is who you are when nobody is watching. If your Hidden Gods are Indirect Wealth, you are privately very interested in the stock market or business, even if your public face is a steady 9-to-5 job.

In [29]:
import json
from lunar_python.util import LunarUtil

def extract_full_luck_timeline(bazi_obj, gender, target_year=2026):
    """
    Extracts 起运, 大运, 流年, and 流月 from a Bazi object.
    :param bazi_obj: The EightChar object
    :param gender: 1 (Male) or 0 (Female)
    :param target_year: The specific year to drill down into (e.g., 2026)
    """
    # 1. Get the Luck (Yun) Object
    # sect=2 uses the more precise minute-based calculation
    luck = bazi_obj.getYun(gender, sect=2)
    dm = bazi_obj.getDayGan()

    # 2. Extract Starting Point (起运)
    payload = {
        "start_point": {
            "dayun_start_time": luck.getStartSolar().toYmdHms(),
            "wait_period": f"{luck.getStartYear()} years, {luck.getStartMonth()} months, {luck.getStartDay()} days"
        },
        "major_luck_decades": []
    }

    # 3. Extract Major Luck Decades (大运)
    da_yun_list = luck.getDaYun()

    # We skip index 0 as it usually represents the period before the first decade starts
    for dy in da_yun_list[1:]:
        decade_pillar = dy.getGanZhi()
        start_yr = dy.getStartYear()
        end_yr = dy.getEndYear()

        decade_data = {
            "pillar": decade_pillar,
            "period": f"{start_yr}-{end_yr}",
            "ten_god_stem": LunarUtil.SHI_SHEN.get(dm + decade_pillar[0]),
            "annual_luck": []
        }

        # 4. Extract Annual Luck (流年) within the target decade
        # Only drill down if the target_year (2026) falls within this decade
        if start_yr <= target_year <= end_yr:
            for ln in dy.getLiuNian():
                # We can filter for 2026 or list all years in the decade
                if ln.getYear() == target_year:
                    annual_pillar = ln.getGanZhi()
                    annual_data = {
                        "year": ln.getYear(),
                        "pillar": annual_pillar,
                        "ten_god": LunarUtil.SHI_SHEN.get(dm + annual_pillar[0]),
                        "monthly_luck": []
                    }

                    # 5. Extract Monthly Luck (流月)
                    for ly in ln.getLiuYue():
                        month_pillar = ly.getGanZhi()
                        annual_data["monthly_luck"].append({
                            "month_name": ly.getMonthInChinese(),
                            "pillar": month_pillar,
                            "ten_god": LunarUtil.SHI_SHEN.get(dm + month_pillar[0])
                        })

                    decade_data["annual_luck"].append(annual_data)

        payload["major_luck_decades"].append(decade_data)

    return payload

# --- Example Usage ---
luck_data = extract_full_luck_timeline(bazi_tst, gender=0)
print(json.dumps(luck_data, indent=2, ensure_ascii=False))

{
  "start_point": {
    "dayun_start_time": "1988-06-14 17:03:17",
    "wait_period": "1 years, 0 months, 11 days"
  },
  "major_luck_decades": [
    {
      "pillar": "丙午",
      "period": "1988-1997",
      "ten_god_stem": "正财",
      "annual_luck": []
    },
    {
      "pillar": "丁未",
      "period": "1998-2007",
      "ten_god_stem": "偏财",
      "annual_luck": []
    },
    {
      "pillar": "戊申",
      "period": "2008-2017",
      "ten_god_stem": "正官",
      "annual_luck": []
    },
    {
      "pillar": "己酉",
      "period": "2018-2027",
      "ten_god_stem": "七杀",
      "annual_luck": [
        {
          "year": 2026,
          "pillar": "丙午",
          "ten_god": "正财",
          "monthly_luck": [
            {
              "month_name": "正",
              "pillar": "庚寅",
              "ten_god": "正印"
            },
            {
              "month_name": "二",
              "pillar": "辛卯",
              "ten_god": "偏印"
            },
            {
              "month_nam

In [30]:
# --- Example Usage ---
luck_data = extract_full_luck_timeline(bazi_std, gender=0)
print(json.dumps(luck_data, indent=2, ensure_ascii=False))

{
  "start_point": {
    "dayun_start_time": "1988-06-09 12:06:00",
    "wait_period": "1 years, 0 months, 6 days"
  },
  "major_luck_decades": [
    {
      "pillar": "丙午",
      "period": "1988-1997",
      "ten_god_stem": "正财",
      "annual_luck": []
    },
    {
      "pillar": "丁未",
      "period": "1998-2007",
      "ten_god_stem": "偏财",
      "annual_luck": []
    },
    {
      "pillar": "戊申",
      "period": "2008-2017",
      "ten_god_stem": "正官",
      "annual_luck": []
    },
    {
      "pillar": "己酉",
      "period": "2018-2027",
      "ten_god_stem": "七杀",
      "annual_luck": [
        {
          "year": 2026,
          "pillar": "丙午",
          "ten_god": "正财",
          "monthly_luck": [
            {
              "month_name": "正",
              "pillar": "庚寅",
              "ten_god": "正印"
            },
            {
              "month_name": "二",
              "pillar": "辛卯",
              "ten_god": "偏印"
            },
            {
              "month_name

The Luck Cycle Logic
Major Luck (大运): These 10-year blocks determine the overarching "season" of your life. If your Day Master is Weak Wood and you enter a 10-year Water cycle, you are being "nourished," making this a decade of growth and support.

Annual Luck (流年): These are the specific year-by-year influences (e.g., 2026 is the year of the Fire Horse). They act as "triggers" for events.

In [31]:
import json
from datetime import datetime, timedelta
from lunar_python import Solar
from lunar_python.util import LunarUtil

def extract_comprehensive_luck(bazi_obj, gender, tst_now):
    """
    Extracts Decade context from EightChar and
    drills down to Day/Hour using the Lunar bridge.
    """
    # 1. High Level Context: The Luck Decade (Yun)
    current_yr = tst_now.getYear()
    # sect=2 for precise minute-based calculation as per your class init
    luck_cycle = bazi_obj.getYun(gender, sect=2)
    dm = bazi_obj.getDayGan()

    payload = {
        "meta": {
            "tst_now": tst_now.toYmdHms(),
            "day_master": dm,
            "gender": "Male" if gender == 1 else "Female"
        },
        "active_decade": {},
        "weekly_forecast": []
    }

    # Find the current 10-year Major Luck (Da Yun)
    for dy in luck_cycle.getDaYun()[1:]:
        if dy.getStartYear() <= current_yr <= dy.getEndYear():
            pillar = dy.getGanZhi()
            payload["active_decade"] = {
                "pillar": pillar,
                "period": f"{dy.getStartYear()}-{dy.getEndYear()}",
                "ten_god": LunarUtil.SHI_SHEN.get(dm + pillar[0])
            }
            break

    # 2. Precision Timing: Next 7 Days (Days & Hours)
    # We step through time starting from the provided TST
    base_dt = datetime.strptime(tst_now.toYmdHms(), "%Y-%m-%d %H:%M:%S")

    for i in range(7):
        target_dt = base_dt + timedelta(days=i)
        # Create solar/lunar objects for the target day
        s = Solar.fromYmdHms(target_dt.year, target_dt.month, target_dt.day,
                             target_dt.hour, target_dt.minute, target_dt.second)
        l = s.getLunar()

        # Use Exact2 for Sect 2 (Midnight-to-Midnight) logic found in your class
        day_pillar = l.getDayInGanZhiExact2()

        day_entry = {
            "date": s.toYmd(),
            "lunar_date": f"{l.getMonthInChinese()}月{l.getDayInChinese()}",
            "day_pillar": day_pillar,
            "ten_god": LunarUtil.SHI_SHEN.get(dm + day_pillar[0]),
            "hours": []
        }

        # Bazi uses 12 double-hours
        for h_idx in range(12):
            # Calculate start of each 2-hour window (0, 2, 4...)
            h_s = Solar.fromYmdHms(target_dt.year, target_dt.month, target_dt.day,
                                   h_idx * 2, 0, 0)
            h_l = h_s.getLunar()
            h_p = h_l.getTimeInGanZhi()

            day_entry["hours"].append({
                "time_slot": h_l.getTimeZhi() + "时",
                "pillar": h_p,
                "ten_god": LunarUtil.SHI_SHEN.get(dm + h_p[0])
            })

        payload["weekly_forecast"].append(day_entry)

    return payload

# --- EXECUTION ---
dt_now = datetime.now()
latitude = 1.4759
longitude = 103.808053

tst_now, _ = get_true_solar_time(dt_now, latitude, longitude)
print(f"Current TST Now: {tst_now.toYmdHms()}, tst_now type: {type(tst_now)}")
result = extract_comprehensive_luck(bazi_tst, gender=0, tst_now=tst_now)
print(json.dumps(result, indent=2, ensure_ascii=False))

Current TST Now: 2026-02-24 12:33:26, tst_now type: <class 'lunar_python.Solar.Solar'>
{
  "meta": {
    "tst_now": "2026-02-24 12:33:26",
    "day_master": "癸",
    "gender": "Female"
  },
  "active_decade": {
    "pillar": "己酉",
    "period": "2018-2027",
    "ten_god": "七杀"
  },
  "weekly_forecast": [
    {
      "date": "2026-02-24",
      "lunar_date": "正月初八",
      "day_pillar": "己巳",
      "ten_god": "七杀",
      "hours": [
        {
          "time_slot": "子时",
          "pillar": "甲子",
          "ten_god": "伤官"
        },
        {
          "time_slot": "丑时",
          "pillar": "乙丑",
          "ten_god": "食神"
        },
        {
          "time_slot": "寅时",
          "pillar": "丙寅",
          "ten_god": "正财"
        },
        {
          "time_slot": "卯时",
          "pillar": "丁卯",
          "ten_god": "偏财"
        },
        {
          "time_slot": "辰时",
          "pillar": "戊辰",
          "ten_god": "正官"
        },
        {
          "time_slot": "巳时",
          "pillar

In [32]:
def analyze_bazi_comprehensive(bazi):
    pillars = ["Year", "Month", "Day", "Hour"]

    # 1. Basic Pillar Info (Stems and Branches)
    stems = [bazi.getYearGan(), bazi.getMonthGan(), bazi.getDayGan(), bazi.getTimeGan()]
    branches = [bazi.getYearZhi(), bazi.getMonthZhi(), bazi.getDayZhi(), bazi.getTimeZhi()]

    # 2. Ten Gods (Stems and Hidden)
    stem_gods = [bazi.getYearShiShenGan(), bazi.getMonthShiShenGan(), "Day Master", bazi.getTimeShiShenGan()]
    hidden_gods = [bazi.getYearShiShenZhi(), bazi.getMonthShiShenZhi(), bazi.getDayShiShenZhi(), bazi.getTimeShiShenZhi()]

    # 3. Strength & Flavor (Life Stages and Na Yin)
    # getDiShi returns the stage of the branch relative to the Day Master
    life_stages = [bazi.getYearDiShi(), bazi.getMonthDiShi(), bazi.getDayDiShi(), bazi.getTimeDiShi()]
    na_yin = [bazi.getYearNaYin(), bazi.getMonthNaYin(), bazi.getDayNaYin(), bazi.getTimeNaYin()]

    print(f"--- Comprehensive Bazi Analysis for {bazi.getDayGan()} Day Master ---")
    header = f"{'Pillar':<8} | {'Pillar':<5} | {'Visible God':<12} | {'Life Stage':<10} | {'Hidden Gods'}"
    print(header)
    print("-" * len(header) * 1)

    for i in range(4):
        p_str = f"{stems[i]}{branches[i]}"
        h_str = ", ".join(hidden_gods[i])
        print(f"{pillars[i]:<8} | {p_str:<5} | {stem_gods[i]:<12} | {life_stages[i]:<10} | {h_str}")

    print("-" * 75)
    # 4. Extracting the "Boss" of the Chart (Monthly Command)
    # This is crucial for career pathing
    print(f"Monthly Command (Focus): {bazi.getMonthShiShenZhi()[0]} (Main)")
    print(f"Melodic Elements (Na Yin): {', '.join(na_yin)}")

# Usage
print("\n--- Comprehensive BaZi Analysis (True Solar Time) ---")
analyze_bazi_comprehensive(bazi_tst)

print("\n--- Comprehensive BaZi Analysis (Standard Time) ---")
analyze_bazi_comprehensive(bazi_std)


--- Comprehensive BaZi Analysis (True Solar Time) ---
--- Comprehensive Bazi Analysis for 癸 Day Master ---
Pillar   | Pillar | Visible God  | Life Stage | Hidden Gods
-----------------------------------------------------------
Year     | 丁卯    | 偏财           | 长生         | 食神
Month    | 乙巳    | 食神           | 胎          | 正财, 正印, 正官
Day      | 癸未    | Day Master   | 墓          | 七杀, 偏财, 食神
Hour     | 戊午    | 正官           | 绝          | 偏财, 七杀
---------------------------------------------------------------------------
Monthly Command (Focus): 正财 (Main)
Melodic Elements (Na Yin): 炉中火, 覆灯火, 杨柳木, 天上火

--- Comprehensive BaZi Analysis (Standard Time) ---
--- Comprehensive Bazi Analysis for 癸 Day Master ---
Pillar   | Pillar | Visible God  | Life Stage | Hidden Gods
-----------------------------------------------------------
Year     | 丁卯    | 偏财           | 长生         | 食神
Month    | 乙巳    | 食神           | 胎          | 正财, 正印, 正官
Day      | 癸未    | Day Master   | 墓          | 七杀, 偏财, 食神
Hou

1. Twelve Life Stages (Di Shi / 十二长生)
This tracks the "Qi" (energy) of an element as it moves through a cycle, similar to the human life cycle. It tells you if a pillar is actually capable of delivering on its promises.

The stages range from peak power to absolute void:

Peak Stages: Chang Sheng (Birth), Guan Dai (Coming of Age), Lin Guan (Career), and Di Wang (Peak/Prosperous). If your Wealth Star is at Di Wang, you have massive earning power.

Declining Stages: Shuai (Weakening), Bing (Sickness), Si (Death), and Jue (Extinct). If your Career Star is at Si or Jue, you may feel stagnant or powerless in your job, even if you have the "right" title.

Resting Stages: Mu (Grave/Storage), Tai (Womb), and Yang (Nurturing).

For you in 2026: You’ll want to check if the 丙午 year sits on a strong stage for your Earth Day Master. (Spoiler: 午 is the "Peak" of Fire, which generates Earth, giving you a very high energy "Resource" boost).

2. Na Yin (纳音 - Melodic Element)
Na Yin is an ancient system that assigns a specific "flavor" of an element to each pair of Stems and Branches. It adds psychological nuance to your personality.

While your standard Bazi says you are 戊土 (Earth), your Na Yin tells us what kind of Earth you are.

Your Day Pillar (戊辰): The Na Yin is "Great Forest Tree" (大林木). This is fascinating because even though your Day Master is Earth, your "Melodic" essence is a powerful, expansive Wood. This suggests that beneath your stable, earthy exterior, you have a deep-seated need for growth, reaching upward, and providing "shade" (protection) for others.

2026 (丙午): The Na Yin is "Heavenly River Water" (天河水). When the "Heavenly River" meets your "Great Forest," it's a year of nourishment.

3. Monthly Command (月令 - Yue Ling)
This is the most powerful spot in your entire chart. The Earthly Branch of your birth month (the Horse, Ox, Tiger, etc.) acts as the "Commanding Officer."

The Monthly Command is determined by which hidden stem was "in charge" on the day you were born.

It defines your "Structure" (格局): It tells us what your "Main Quest" in life is.

In your case: You were born in the month of 亥 (Pig). This month is dominated by 壬水 (Indirect Wealth).

The Result: Even though you have many "Resource" (Support) stars visible, your Internal Command is driven by Wealth. You aren't just a scholar; you are a scholar who knows how to spot market trends and financial opportunities.

How to add this to your Python logic:
To extract these, you would use:

Life Stages: bazi.getYearDiShi(), bazi.getMonthDiShi(), etc.

Na Yin: bazi.getYearNaYin(), bazi.getMonthNaYin(), etc.

Monthly Command: This is usually the main hidden stem of the month branch: bazi.getMonthShiShenZhi()[0].

In [33]:
def getYearLu(baZi):
    """
    Custom function to get Year Pillar Prosperity (年禄)
    """
    # 1. Get the Year Stem and Branch from your EightChar object
    year_gan = baZi.getYearGan()
    year_zhi = baZi.getYearZhi()

    # 2. Look up the Stem's Prosperity Branch in the library's utility
    # LunarUtil.LU is the dictionary mapping Stems to their Lu Branches
    gan_lu_branch = LunarUtil.LU.get(year_gan)

    # 3. Check if the Year Branch itself is also a Lu Branch
    # (Some special configurations use this for "Entering Prosperity")
    zhi_lu = None
    if year_zhi in LunarUtil.LU:
        zhi_lu = LunarUtil.LU.get(year_zhi)

    # 4. Construct the string using the library's naming convention
    # "Mutual Prosperity" (互禄) is the standard term for this mapping
    lu_string = gan_lu_branch + "命互禄"

    if zhi_lu is not None:
        lu_string += " " + zhi_lu + "命进禄"

    return lu_string

In [34]:
from lunar_python import Lunar, Solar  # Import Lunar and Solar date conversion classes
from lunar_python.util import HolidayUtil  # Import utility for holiday information
from datetime import datetime  # Import datetime for current date/time

# Corinne's birthday example
# solar_birthday= Solar.fromYmdHms(1987, 6, 3, 12, 6, 0)  # Create solar date June 3, 1987 at 12:06 PM
# tst_birthday, inputs_report = get_true_solar_time(datetime(1987, 6, 3, 12, 6, 0), 1.4759, 103.808053)  # Get true solar time for the birthday

# Desmond's birthday example
solar_birthday= Solar.fromYmdHms(1985, 11, 25, 17, 7, 0)  # Create solar date
tst_birthday, inputs_report = get_true_solar_time(datetime(1985, 11, 25, 17, 7, 0), 1.3253, 103.8415)  # Get true solar time for the birthday

# Lara's birthday example
# solar_birthday= Solar.fromYmdHms(2025, 7, 31, 9, 10, 0)  # Create solar date
# tst_birthday, inputs_report = get_true_solar_time(datetime(2025, 7, 31, 9, 10, 0), 1.3253, 103.8415)  # Get true solar time for the birthday

print('阳历生日: ' + solar_birthday.toYmdHms())  # Print solar birthday
print('真太阳时生日: ' + tst_birthday.toYmdHms())  # Print true solar time birthday
print(f'Input Details for TST Calculation: {inputs_report}')  # Print header for input details

# 节气表 Jiéqì Biǎo Solar terms (24 seasonal division points)
lunar_birthday = tst_birthday.getLunar()  # Convert true solar time birthday to lunar calendar
print('农历生日: ' + lunar_birthday.toFullString())  # Print lunar birthday in full string format
jieQi_table = lunar_birthday.getJieQiTable()  # Get all 24 solar terms with their dates

print('节气表 (Solar Terms):')  # Print header for solar terms
for k in lunar_birthday.getJieQiList():  # Loop through each solar term name
    print(k + ' = ' + jieQi_table[k].toYmdHms())  # Print solar term name and exact date/time
print('')  # Print empty line for formatting

# 八字 Bāzì Eight Character (Four Pillars of Destiny)
print('八字 (BaZi):')  # Print header for Eight Characters
baZi = lunar_birthday.getEightChar()  # Extract the 8-character fortune reading from lunar date
# Display year, month, day, and hour pillars
print(f"年柱: 天干 {baZi.getYearGan()} | 地支 {baZi.getYearZhi()}")
print(f"月柱: 天干 {baZi.getMonthGan()} | 地支 {baZi.getMonthZhi()}")
print(f"日柱: 天干 {baZi.getDayGan()} | 地支 {baZi.getDayZhi()}")
print(f"时柱: 天干 {baZi.getTimeGan()} | 地支 {baZi.getTimeZhi()}")
print('')  # Print empty line for formatting

# 八字五行 bā zì wǔ xíng Five Elements of the Bazi
print('五行 (Five Elements):')  # Print header for Five Elements
# Print the 5-element composition (Wood, Fire, Earth, Metal, Water)
print(f"年柱五行: {baZi.getYearWuXing()}")
print(f"月柱五行: {baZi.getMonthWuXing()}")
print(f"日柱五行: {baZi.getDayWuXing()}")
print(f"时柱五行: {baZi.getTimeWuXing()}")
print('')  # Print empty line for formatting

# 八字天干十神 bā zì tiān gān shí shén Ten Gods of heavenly stems (represents destiny patterns)
print('天干十神 (Ten Gods of Heavenly Stems):')  # Print header for Ten Gods of heavenly stems
# Show fortune interpretation for each pillar stem
print(f"年干十神: {baZi.getYearShiShenGan()}")
print(f"月干十神: {baZi.getMonthShiShenGan()}")
print(f"日干十神: {baZi.getDayShiShenGan()}") # This will usually say '日主' or '元神'
print(f"时干十神: {baZi.getTimeShiShenGan()}")
print('')  # Print empty line for formatting

# 八字纳音 bā zì nà yīn Melodic Elements of the Bazi
# print('纳音 (Na Yin - Melodic Elements):')  # Print header for Melodic Elements

# 12 Earthly Branches Wu Xing Elements - this is a more detailed breakdown of the branch elements, which can differ from the stem elements and provide additional insight into the chart's flavor.
# Is there one?
# print('地支五行 (Earthly Branches Five Elements):')  # Print header for Earthly Branches Five Elements
# print(f"年支五行: {baZi.getYearZhiWuXing()}")
# print(f"月支五行: {baZi.getMonthZhiWuXing()}")
# print(f"日支五行: {baZi.getDayZhiWuXing()}")
# print(f"时支五行: {baZi.getTimeZhiWuXing()}")
# print('')  # Print empty line for formatting

NAYIN_YIXIANG_MAP = {
    "海中金": "潜藏于深海，宁静而深邃",
    "剑锋金": "淬火成锋，刚烈决断",
    "白蜡金": "柔中带刚，如蜡质般温润反光",
    "沙中金": "淘金于沙，需经磨炼方显贵气",
    "金箔金": "轻薄优雅，饰物之华美装饰",
    "钗钏金": "首饰之金，温婉柔和",
    "大林木": "森林茂盛，生机勃勃",
    "杨柳木": "柳丝轻拂，随风而动，灵活柔顺",
    "松柏木": "凌霜傲雪，坚毅不拔之志",
    "平地木": "平原之上，稳步扎根生长",
    "桑柘木": "质地坚韧，实用且长久",
    "石榴木": "繁花似锦，果实累累，充满活力",
    "涧下水": "山间溪流，活泼跳动",
    "大溪水": "奔流不息，气势开阔",
    "长流水": "绵延不绝，智慧与适应力",
    "天河水": "银河落九天，纯净而高远",
    "泉中水": "井泉清澈，源源不断，内敛智慧",
    "大海水": "汪洋浩瀚，包容万物之势",
    "炉中火": "炉火温旺，坚持不懈，极具创造力",
    "山头火": "峰顶赤光，显赫可见",
    "霹雳火": "电闪雷鸣，爆发力极强",
    "山下火": "谷底微光，温和而易控",
    "覆灯火": "佛灯长明，宁静且带有灵性",
    "天上火": "烈日当空，纯净的照明力量",
    "路旁土": "坚实肥沃，承载行人往来",
    "城头土": "稳固坚实，守护防御之力",
    "屋上土": "遮风挡雨，历经风暴仍可靠",
    "壁上土": "结构支撑，脚踏实地的毅力",
    "大驿土": "通达四方，连接与支撑",
    "沙中土": "散而不乱，随方就圆，灵活而有目的"
}

# 纳音 Na Yin
print('纳音 (Na Yin):')  # Print header for Na Yin
# Lookup and print Na Yin with 意象
for pillar, get_na_yin in [
    ("年柱", lunar_birthday.getYearNaYin),
    ("月柱", lunar_birthday.getMonthNaYin),
    ("日柱", lunar_birthday.getDayNaYin),
    ("时柱", lunar_birthday.getTimeNaYin)
]:
    nayin = get_na_yin()
    yixiang = NAYIN_YIXIANG_MAP.get(nayin, "无意象")
    print(f"{pillar}纳音: {nayin} | 意象: {yixiang}")
print('')  # Print empty line for formatting

"""
The "Golden Format" for the LLM
Use a table or a clear list that pairs the Pillar with its Melodic Element. This structure is highly scannable for AI:
柱位 (Pillar)	干支 (GanZhi)	十神 (Ten God)	纳音 (Na Yin)	意象 (Symbolic Meaning)
年柱 (Year)	乙丑	正官	覆灯火	照亮黑暗的灯火 (Lantern Fire)
月柱 (Month)	丁亥	正印	松柏木	傲雪凌霜的劲松 (Cypress Wood)
日柱 (Day)	戊辰	日主	沙中土	蕴含矿石的细沙 (Sand Earth)
时柱 (Hour)	庚申	食神	平地木	广阔平原的萌芽 (Plain Wood)
"""
# 十二长生 地势 Di Shi 12 Life Stages
print(f'地势 Di Shi 12 Life Stages')
print(f"年柱地势: {baZi.getYearDiShi()}")
print(f"月柱地势: {baZi.getMonthDiShi()}")
print(f"日柱地势: {baZi.getDayDiShi()}")
print(f"时柱地势: {baZi.getTimeDiShi()}")

# # 十二律吕 shí èr lǜ lǚ Twelve Lü (Musical Notes) Not available in library.
# print('十二律吕 (Twelve Lü - Musical Notes):')  # Print header for
# print(f"年柱十二律吕: ")
# print(f"年柱十二律吕: {lunar_birthday.getYearLu()}")
# print(f"月柱十二律吕: {lunar_birthday.getMonthLu()}")
# print(f"日柱十二律吕: {lunar_birthday.getDayLu()}")
# print(f"时柱十二律吕: {lunar_birthday.getTimeLu()}")
# print('')  # Print empty line for formatting

# Day Prosperity (日禄)
print('日禄 (Day Prosperity):')  # Print header for Day Prosperity
day_lu = lunar_birthday.getDayLu()  # Get Day Prosperity from lunar birthday
print(f"日柱日禄: {day_lu}")  # Print Day Prosperity
print('')  # Print empty line for formatting

# 八字地支十神 bā zì dì zhī shí shén Ten Gods of earthly branches (internal influences)
print('地支十神 (Ten Gods of Earthly Branches):')  # Print header for Ten Gods of earthly branches
# Display first hidden influence for each branch
def format_hidden_gods(gods_list):
    labels = ["本气 (Main Qi)", "中气 (Mid Qi)", "余气 (Residual Qi)"]
    # Combine label with the god name, only for as many as exist in the list
    return ", ".join([f"{labels[i]}:{gods_list[i]}" for i in range(len(gods_list))])

print(f"年支藏干十神: {format_hidden_gods(baZi.getYearShiShenZhi())}")
print(f"月支藏干十神: {format_hidden_gods(baZi.getMonthShiShenZhi())}")
print(f"日支藏干十神: {format_hidden_gods(baZi.getDayShiShenZhi())}")
print(f"时支藏干十神: {format_hidden_gods(baZi.getTimeShiShenZhi())}")
print('')  # Print empty line for formatting

print("--- 旬空与五行循环 (Xun Kong & Cycles) ---")
# Use the "Exact" methods to ensure the LLM gets the most precise calculation
print(f"年柱 (Year): {lunar_birthday.getYearInGanZhiExact()} | 旬: {lunar_birthday.getYearXunExact()} | 旬空: {lunar_birthday.getYearXunKongExact()}")
print(f"月柱 (Month): {lunar_birthday.getMonthInGanZhiExact()} | 旬: {lunar_birthday.getMonthXunExact()} | 旬空: {lunar_birthday.getMonthXunKongExact()}")
# getDayXunExact2() is used specifically for the Day Pillar. new day starts at Night Zi (夜子时), which is 11:00 PM (23:00), not 12:00 AM. for Sect 2 (midnight-to-midnight) logic
print(f"日柱 (Day): {lunar_birthday.getDayInGanZhiExact()} | 旬: {lunar_birthday.getDayXunExact2()} | 旬空: {lunar_birthday.getDayXunKongExact2()}")
# Note: Ensure your library supports getTimeXunExact if uncommenting
print(f"时柱 (Hour): {lunar_birthday.getTimeInGanZhi()} | 旬: {lunar_birthday.getTimeXun()} | 旬空: {lunar_birthday.getTimeXunKong()}")
print(" ")

# 三才 - Sān Cái Three Talents (Heaven, Earth, Human) - This is a fundamental concept in BaZi that describes the overall balance of the chart. It can be used to quickly assess the general "flavor" of the destiny.
print(f"三才 (Three Talents)")
print("--- 三垣分析 (Three Palaces Analysis) ---")
print(f"胎元 (Conception): {baZi.getTaiYuan()} | 纳音: {baZi.getTaiYuanNaYin()}")
print(f"命宫 (Life Palace): {baZi.getMingGong()} | 纳音: {baZi.getMingGongNaYin()}")
print(f"身宫 (Action Palace): {baZi.getShenGong()} | 纳音: {baZi.getShenGongNaYin()}")
print(f"胎息 (Embryonic Breath): {baZi.getTaiXi()} | 纳音: {baZi.getTaiXiNaYin()}")
print(" ")

# 时柱 - Future & Children
print(f"时柱 (The Result): {baZi.getTime()}")
print(f"时柱五行: {baZi.getTimeWuXing()}")
print(f"时柱纳音: {baZi.getTimeNaYin()}")
print(f"时柱地势: {baZi.getTimeDiShi()}")
print(" ")

# 日柱 - The Core
print(f"日柱 (The Self): {baZi.getDay()}")
print(" ")

print("--- 环境与历法细节 (Environmental & Almanac Details) ---")
# 三伏 (San Fu - Dog Days of Summer)
# Since you were born in Nov, this will likely be None (非伏天)
fu = lunar_birthday.getFu()
print(f"三伏 (Three Fu): {fu if fu else '非三伏天'}")

# Extract the "Shu Jiu" status (Winter energy)
shujiu = lunar_birthday.getShuJiu()
print(f"数九 (Winter Stage): {shujiu if shujiu else '非数九天 (Pre-Solstice)'}")

# 六曜 (Liu Yao - The Six Bright Stars)
print(f"六曜 (Liu Yao): {lunar_birthday.getLiuYao()}")

# 物候 与 候 (Wu Hou & Hou - Phenology)
# Tells you exactly what nature was doing (e.g., "Water begins to freeze")
print(f"物候 (Phenology): {lunar_birthday.getWuHou()}")
print(f"候 (Seasonal Phase): {lunar_birthday.getHou()}")

# 日禄 (Day Prosperity / Lu)
# This is your '巳命互禄' wealth marker
print(f"日禄 (Day Prosperity): {lunar_birthday.getDayLu()}")

# 佛历 与 道历 (Buddhist & Taoist Calendars)
# Useful for spiritual resonance analysis
foto = lunar_birthday.getFoto()
tao = lunar_birthday.getTao()
print(f"佛历 (Buddhist Calendar): {foto.toFullString()}")
print(f"道历 (Taoist Calendar): {tao.toFullString()}")

# 时辰 (Lunar Time)
# Confirms the exact double-hour of birth
print(f"出生时辰 (Lunar Time): {lunar_birthday.getTime().getGanZhi()}时")

print("--- 综合天命背景 (Comprehensive Destiny Background) ---")
# # The Full Almanac String (Recheck for repeat)
# print("Alamac Summary")
# print(f"全称描述: {lunar_birthday.toFullString()}")

print("The information on the day i was born. ")
# 1. Base Information
print(f"农历日期: {lunar_birthday.toString()}")
print(f"星期: {lunar_birthday.getWeekInChinese()}")

# 2. Advanced Astro-Geomancy (Properly Labeled)
print(f"二十八星宿: {lunar_birthday.getXiu()}{lunar_birthday.getZheng()}{lunar_birthday.getAnimal()} ({lunar_birthday.getXiuLuck()})")
print(f"方位方位 (Directional Quadrant): {lunar_birthday.getGong()}方{lunar_birthday.getShou()}")

# 3. Ritual & Selection (Properly Labeled)
print(f"彭祖百忌 (Taboos): {lunar_birthday.getPengZuGan()}，{lunar_birthday.getPengZuZhi()}")
print(f"日冲 (Daily Clash): {lunar_birthday.getChongDesc()}")
print(f"日煞 (Daily Sha): {lunar_birthday.getSha()}")

# 4. Strategic Directions (The "Compass")
print("方位 - Where the georaphic luck is located")
print(f"财神方位 (Wealth Direction): {lunar_birthday.getDayPositionCaiDesc()} ({lunar_birthday.getDayPositionCai()})")
print(f"喜神方位 (Joy Direction): {lunar_birthday.getDayPositionXiDesc()} ({lunar_birthday.getDayPositionXi()})")
print(f"福神方位 (Fortune Direction): {lunar_birthday.getDayPositionFuDesc()} ({lunar_birthday.getDayPositionFu()})")
print(f"阳贵神方位 (Yang Noble Direction): {lunar_birthday.getDayPositionYangGuiDesc()} ({lunar_birthday.getDayPositionYangGui()})")
print(f"阴贵神方位 (Yin Noble Direction): {lunar_birthday.getDayPositionYinGuiDesc()} ({lunar_birthday.getDayPositionYinGui()})")
print(" ")

print("天神 Deity")
print(f"日值天神: {lunar_birthday.getDayTianShen()} ({lunar_birthday.getDayTianShenLuck()})")
print(f"时值天神: {lunar_birthday.getTimeTianShen()} ({lunar_birthday.getTimeTianShenLuck()})")

# 宜/忌 (Auspicious & Inauspicious Actions)
print(f"每日宜: {lunar_birthday.getDayYi()}")
print(f"每日忌: {lunar_birthday.getDayJi()}")
print(f"吉神宜趋: {lunar_birthday.getDayJiShen()}")
print(f"凶煞宜忌: {lunar_birthday.getDayXiongSha()}")
print(f"月相 (Moon Phase): {lunar_birthday.getYueXiang()}")
print(' ')

# 二十八星宿
print("--- 星宿与神性背景 (Constellation & Spiritual Background) ---")
print(f"星宿: {lunar_birthday.getXiu()}{lunar_birthday.getZheng()}{lunar_birthday.getAnimal()} ({lunar_birthday.getXiuLuck()})")
print(f"星宿诗诀: {lunar_birthday.getXiuSong()}") # The traditional poem
print(f"方位 (Palace/Animal): {lunar_birthday.getGong()}方{lunar_birthday.getShou()}")
print(f"传统节日: {lunar_birthday.getFestivals()} {lunar_birthday.getOtherFestivals()}")
print(' ')
print("--- 冲突与季节背景 (Clashes & Seasonal Context) ---")
print(f"日柱冲克: {lunar_birthday.getDayChongDesc()}")
print(f"日柱方位煞: {lunar_birthday.getDaySha()}")
print(f"生于季节: {lunar_birthday.getSeason()}")
print(' ')

print("--- 节气深度分析 (Solar Term Depth Analysis) ---")
# Get the previous Section (The start of your birth month)
prev_jie = lunar_birthday.getPrevJie()
print(f"入节 (Start of Month): {prev_jie.getName()} | 时间: {prev_jie.getSolar().toYmdHms()}")

# Get the next Section (The end of your birth month)
next_jie = lunar_birthday.getNextJie()
print(f"下个节令 (Next Month Start): {next_jie.getName()} | 时间: {next_jie.getSolar().toYmdHms()}")

# Get the current Qi (The midpoint of the month energy)
prev_qi = lunar_birthday.getPrevQi()
print(f"气令 (Mid-month Marker): {prev_qi.getName()} | 时间: {prev_qi.getSolar().toYmdHms()}")
print(' ')

# 胎元 - Tāi Yuán Conception Palace - This represents the prenatal environment and can provide insights into inherited traits and early life influences.
print(f"胎元 (Conception Palace): {baZi.getTaiYuan()}")
print(f"胎元纳音 (Conception Palace Soul): {baZi.getTaiYuanNaYin()}")
print(" ")

# 命宫 - Mìng Gōng Life Palace - This indicates the overall life path and destiny, often used to assess career and life purpose.
print(f"命宫 (Life Palace): {baZi.getMingGong()}")
print(f"命宫纳音 (Life Palace Soul): {baZi.getMingGongNaYin()}")
print(" ")

# 身宫 - Shēn Gōng Action Palace - This reflects the physical body and actions, shedding light on health and how one interacts with the world.
# 2. Extract the Life Palace (Shen Gong)
print(f"身宫 (Body Palace): {baZi.getShenGong()}")
print(f"身宫纳音 (Body Palace Soul): {baZi.getShenGongNaYin()}")
print(" ")

# 9 Star: Params: (1=Lunar New Year, 2=Li Chun, 3=Solar Term. Exact minute and second based on Li Chun transition)
print("--- 九星能量 (Nine Star Energy / Feng Shui) ---")
# Year Star (Calculated by Li Chun boundary)
print(f"年九星 (Year Star): {lunar_birthday.getYearNineStar(3).toFullString()}")

# Month Star
print(f"月九星 (Month Star): {lunar_birthday.getMonthNineStar(3).toFullString()}")

# Day Star (Determined by Solstice proximity)
print(f"日九星 (Day Star): {lunar_birthday.getDayNineStar().toFullString()}")

# Time Star
print(f"时九星 (Time Star): {lunar_birthday.getTimeNineStar().toFullString()}")
print(' ')

# Assuming 'lunar_birthday' is your object
print("--- 出生时刻方位 (Birth Moment Compass) ---")

# Hour-specific Luck Directions
print(f"时分财神方位: {lunar_birthday.getTimePositionCaiDesc()} ({lunar_birthday.getTimePositionCai()})")
print(f"时分喜神方位: {lunar_birthday.getTimePositionXiDesc()} ({lunar_birthday.getTimePositionXi()})")
print(f"时分福神方位: {lunar_birthday.getTimePositionFuDesc()} ({lunar_birthday.getTimePositionFu()})")
print(f"时分阳贵人: {lunar_birthday.getTimePositionYangGuiDesc()} ({lunar_birthday.getTimePositionYangGui()})")
print(f"时分阴贵人: {lunar_birthday.getTimePositionYinGuiDesc()} ({lunar_birthday.getTimePositionYinGui()})")

# 年太岁 Year Tai Sui position (relative to Tai Sui deity, 12 positions)
print("--- 太岁位置 (Tai Sui Positions) ---")
print(f"年太岁位置: {lunar_birthday.getYearPositionTaiSui()}")  # Print year Tai Sui position
print(f"年太岁描述: {lunar_birthday.getYearPositionTaiSuiDesc()}")  # Print year Tai Sui position description

# 月太岁 Month Tai Sui position (relative to Tai Sui deity, 12 positions)
print(f"月太岁位置: {lunar_birthday.getMonthPositionTaiSui()}")  # Print month Tai Sui position
print(f"月太岁描述: {lunar_birthday.getMonthPositionTaiSuiDesc()}")  # Print month Tai Sui position description

# 日太岁 Day Tai Sui position (relative to Tai Sui deity, 12 positions)
print(f"日太岁位置: {lunar_birthday.getDayPositionTaiSui()}")  # Print day Tai Sui position
print(f"日太岁描述: {lunar_birthday.getDayPositionTaiSuiDesc()}")  # Print day Tai Sui position description

# # Zhi Shen (Branch Stars) - The specific "Vibe" of the pillar
# # These often include things like "Tian Yi" or "Wen Chang" in some versions
# print(f"年支神煞: {baZi.getYearZhiShen()}")
# print(f"月支神煞: {baZi.getMonthZhiShen()}")
# print(f"日支神煞: {baZi.getDayZhiShen()}")
# print(f"时支神煞: {baZi.getTimeZhiShen()}")

# 八字神煞 bā zì shén shà Special Stars (mythical influences)
# print('神煞 (Special Stars):')  # Print header for Special Stars
# print(f"年柱神煞 (Year Stars): {lunar_birthday.getYearShenSha()}")
# print(f"月柱神煞 (Month Stars): {lunar_birthday.getMonthShenSha()}")
# print(f"日柱神煞 (Day Stars): {lunar_birthday.getDayShenSha()}")
# print(f"时柱神煞 (Hour Stars): {lunar_birthday.getTimeShenSha()}")

# 女运 Female luck cycle (starts from age 0)
yun = baZi.getYun(0)  # Calculate luck cycle starting from birth (0=female, 1=male)
print('起运 (Luck Cycle Start):')  # Print header for luck cycle start
start_solar = yun.getStartSolar() # Gets the Solar date the luck starts
print(f"起运时间: {start_solar.toYmdHms()}") # Print exact date/time luck starts
print('')  # Print empty line for formatting

# 大运 Major luck cycles (10-year periods)
print('大运 (Major Luck Cycles):')  # Print header for major luck cycles
daYunArr = yun.getDaYun()

for i in range(1, len(daYunArr)):
    dy = daYunArr[i]
    gan_zhi = dy.getGanZhi()
    gan = gan_zhi[0]
    zhi = gan_zhi[1]

    # 1. Calculate Stem Ten God
    gan_shishen = LunarUtil.SHI_SHEN.get(baZi.getDayGan() + gan)

    # 2. Calculate Hidden Branch Ten Gods
    hide_gan = LunarUtil.ZHI_HIDE_GAN.get(zhi)
    zhi_shishen_list = [LunarUtil.SHI_SHEN.get(baZi.getDayGan() + h_gan) for h_gan in hide_gan]

    # 3. Format with Professional 3-Tier Labels
    # We map the results to: [本气 (Main), 中气 (Mid), 余气 (Residual)]
    labels = ["本气 (Main Qi)", "中气 (Mid Qi)", "余气 (Residual Qi)"]
    formatted_parts = []

    for idx, shishen in enumerate(zhi_shishen_list):
        if idx < len(labels):
            formatted_parts.append(f"{labels[idx]}:{shishen}")

    branch_info = " | ".join(formatted_parts)

    print(f"大运[{i}] {dy.getStartYear()}年 | {gan_zhi} | 天干十神: {gan_shishen}")
    print(f"       地支藏干: [{branch_info}]")

print('')  # Print empty line for formatting

# 大运[0] 流年 Annual luck for major cycle 0 (year-by-year within the decade)
print('流年 (Annual Luck):')  # Print header for annual luck
print('Need to code this to retrieve the last 5 year + next 5 years')
current_da_yun = daYunArr[4]
liuNianArr = current_da_yun.getLiuNian()

print(f"--- {current_da_yun.getGanZhi()} 大运中的流年分析 (Current Window) ---")

for ln in liuNianArr:
    # Filter for relevant years (e.g., 2024 to 2028)
    if 2025 <= ln.getYear() <= 2029:
        gz = ln.getGanZhi()
        gan, zhi = gz[0], gz[1]

        # 1. Stem Ten God
        gan_ss = LunarUtil.SHI_SHEN.get(baZi.getDayGan() + gan)

        # 2. Branch Hidden Ten Gods (Main/Mid/Res)
        h_gans = LunarUtil.ZHI_HIDE_GAN.get(zhi)
        ss_list = [LunarUtil.SHI_SHEN.get(baZi.getDayGan() + h) for h in h_gans]

        labels = ["本气(Main)", "中气(Mid)", "余气(Res)"]
        branch_ss = " | ".join([f"{labels[i]}:{ss_list[i]}" for i in range(len(ss_list))])

        print(f"流年 {ln.getYear()}年 ({ln.getAge()}岁) | {gz} | 天干:{gan_ss}")
        print(f"       地支藏干: [{branch_ss}]")

# 大运[0] 小运 Minor luck for major cycle 0 (sub-cycles within decade)
print('小运 (Minor Luck):')  # Print header for minor luck
xiaoYunArr = daYunArr[0].getXiaoYun()  # Get all minor cycles within first major period
for i in range(0, len(xiaoYunArr)):  # Loop through each minor cycle
    xiaoYun = xiaoYunArr[i]  # Get current minor cycle
    print('小运[' + str(i) + '] ' + str(xiaoYun.getYear()) + '年 ' + str(xiaoYun.getAge()) + '岁 ' + xiaoYun.getGanZhi())  # Print cycle index, year, age, and Heavenly Stem/Earthly Branch
print('')  # Print empty line for formatting

# 流年[0] 流月 Monthly luck for first annual cycle (month-by-month within the year)
liuYueArr = liuNianArr[0].getLiuYue()  # Get all monthly cycles within first annual period
for i in range(0, len(liuYueArr)):  # Loop through each month
    liuYue = liuYueArr[i]  # Get current monthly cycle
    print('流月[' + str(i) + '] ' + str(liuYue.getMonthInChinese()) + '月 ' + liuYue.getGanZhi())  # Print month index, Chinese month name, and Heavenly Stem/Earthly Branch
print('')  # Print empty line for formatting

print('--- 十二时辰 五鼠遁 (Twelve Traditional Chinese Double-Hour Periods) Five Rat Method ---')
times = lunar_birthday.getTimes()  # Get array of 12 traditional Chinese double-hour time periods
for i in range(0, len(times)):  # Loop through each time period
    time = times[i]  # Get current time period object
    print("%s - %s : %s" % (time.getMinHm(), time.getMaxHm(), time.toString()))  # Print time period range (start-end) and period name


阳历生日: 1985-11-25 17:07:00
真太阳时生日: 1985-11-25 16:14:36
Input Details for TST Calculation: {'datetime': datetime.datetime(1987, 6, 3, 12, 6), 'latitude': 1.3253, 'longitude': 103.8415, 'standard_meridian': 120.0, 'day_of_year': 329, 'eot_minutes': 12.232695484260729, 'longitude_correction': -64.63400000000001, 'total_adjustment_minutes': -52.401304515739284}
农历生日: 一九八五年十月十四 乙丑(牛)年 丁亥(猪)月 戊辰(龙)日 申(猴)时 纳音[海中金 屋上土 大林木 石榴木] 星期一 西方白虎 星宿[毕月乌](吉) 彭祖百忌[戊不受田田主不祥 辰不哭泣必主重丧] 喜神方位[巽](东南) 阳贵神方位[艮](东北) 阴贵神方位[坤](西南) 福神方位[艮](东北) 财神方位[坎](正北) 冲[(壬戌)狗] 煞[南]
节气表 (Solar Terms):
DA_XUE = 1984-12-07 06:28:03
冬至 = 1984-12-22 00:22:48
小寒 = 1985-01-05 17:35:05
大寒 = 1985-01-20 10:57:33
立春 = 1985-02-04 05:11:47
雨水 = 1985-02-19 01:07:21
惊蛰 = 1985-03-05 23:16:21
春分 = 1985-03-21 00:13:43
清明 = 1985-04-05 04:13:35
谷雨 = 1985-04-20 11:25:46
立夏 = 1985-05-05 21:42:32
小满 = 1985-05-21 10:42:55
芒种 = 1985-06-06 01:59:56
夏至 = 1985-06-21 18:44:07
小暑 = 1985-07-07 12:18:35
大暑 = 1985-07-23 05:36:26
立秋 = 1985-08-07 22:04:16
处暑 = 1985-

In [37]:
ZHI_TO_HOUR_NAME = {
        "子": "Zi", "丑": "Chou", "寅": "Yin", "卯": "Mao",
        "辰": "Chen", "巳": "Si", "午": "Wu", "未": "Wei",
        "申": "Shen", "酉": "You", "戌": "Xu", "亥": "Hai"
    }

# Bone weights for years, months, days, and hours based on the Yuan Tian Gang system
YUAN_TIAN_GANG_BONE_WEIGHTS = {
    "years": {
        0: 1.2, 1: 0.9, 2: 0.6, 3: 0.7, 4: 1.2, 5: 0.4, 6: 0.9, 7: 0.8, 8: 0.7, 9: 0.8,
        10: 1.5, 11: 0.9, 12: 1.6, 13: 0.8, 14: 0.8, 15: 1.9, 16: 1.2, 17: 0.6, 18: 0.8, 19: 0.7,
        20: 0.5, 21: 1.5, 22: 0.6, 23: 1.6, 24: 0.7, 25: 0.8, 26: 0.9, 27: 0.7, 28: 1.0, 29: 0.7,
        30: 1.5, 31: 0.6, 32: 0.5, 33: 1.4, 34: 1.4, 35: 0.9, 36: 0.7, 37: 0.7, 38: 0.9, 39: 1.2,
        40: 0.8, 41: 0.7, 42: 1.3, 43: 0.5, 44: 1.4, 45: 0.5, 46: 1.9, 47: 1.7, 48: 0.5, 49: 0.7,
        50: 1.2, 51: 0.8, 52: 0.8, 53: 0.6, 54: 1.9, 55: 0.6, 56: 0.8, 57: 0.5, 58: 1.0, 59: 0.7
    },
    "months": {
        1: 0.6, 2: 0.7, 3: 1.8, 4: 0.9, 5: 0.5, 6: 1.6,
        7: 0.9, 8: 1.5, 9: 1.5, 10: 0.8, 11: 0.9, 12: 0.5
    },
    "days": {
        1: 0.5, 2: 1.0, 3: 0.8, 4: 1.5, 5: 1.6, 6: 1.5, 7: 0.8, 8: 1.6, 9: 0.8, 10: 1.6,
        11: 0.9, 12: 1.7, 13: 0.8, 14: 1.7, 15: 1.0, 16: 0.8, 17: 0.9, 18: 1.8, 19: 0.5, 20: 1.5,
        21: 1.0, 22: 0.9, 23: 0.8, 24: 0.9, 25: 1.5, 26: 1.8, 27: 0.7, 28: 0.8, 29: 1.6, 30: 0.6
    },
    "hours": {
        "Zi": 1.6, "Chou": 0.6, "Yin": 0.7, "Mao": 1.0, "Chen": 0.9, "Si": 1.6,
        "Wu": 1.0, "Wei": 0.8, "Shen": 0.8, "You": 0.9, "Xu": 0.6, "Hai": 0.6
    }
}

def calculate_yuan_tian_gang_bone_weight(lunar_birthday):
    """
    Calculate the Yuan Tian Gang bone weight based on lunar birthday.

    Args:
        lunar_birthday: A Lunar object with lunar date information
        bone_weights: Dictionary with 'years', 'months', 'days', 'hours' keys containing weight mappings

    Returns:
        A dictionary with breakdown and total bone weight
    """

    # 1. Extract Year and map to 0-59 range (60-year sexagenary cycle)
    # Starting reference: 1984 = Index 0 (Jiazi 甲子 - Rat Year)
    year = lunar_birthday.getYear()
    year_index = (year - 1924) % 60
    if year_index not in YUAN_TIAN_GANG_BONE_WEIGHTS["years"]:
        raise ValueError(f"Year index {year_index} (year {year}) not found in YUAN_TIAN_GANG_BONE_WEIGHTS['years']")
    year_weight = YUAN_TIAN_GANG_BONE_WEIGHTS["years"][year_index]

    # 2. Extract Month (1-12)
    month = lunar_birthday.getMonth()
    if month not in YUAN_TIAN_GANG_BONE_WEIGHTS["months"]:
        raise ValueError(f"Month {month} not found in bone_weights['months']")
    month_weight = YUAN_TIAN_GANG_BONE_WEIGHTS["months"][month]

    # 3. Extract Day (1-30)
    day = lunar_birthday.getDay()
    if day not in YUAN_TIAN_GANG_BONE_WEIGHTS["days"]:
        raise ValueError(f"Day {day} not found in YUAN_TIAN_GANG_BONE_WEIGHTS['days']")
    day_weight = YUAN_TIAN_GANG_BONE_WEIGHTS["days"][day]

    # 4. Extract Hour and convert to Western name
    # Get the earthly branch (Zhi) of the hour from the BaZi eight character
    hour_zhi = lunar_birthday.getEightChar().getTimeZhi()
    if hour_zhi not in ZHI_TO_HOUR_NAME:
        raise ValueError(f"Hour earthly branch '{hour_zhi}' not recognized")
    hour_name = ZHI_TO_HOUR_NAME[hour_zhi]
    if hour_name not in YUAN_TIAN_GANG_BONE_WEIGHTS["hours"]:
        raise ValueError(f"Hour '{hour_name}' not found in bone_weights['hours']")
    hour_weight = YUAN_TIAN_GANG_BONE_WEIGHTS["hours"][hour_name]

    # 5. Calculate total bone weight
    total_weight = year_weight + month_weight + day_weight + hour_weight

    return {
        "lunar_date": lunar_birthday.toString(),
        "year": year,
        "year_index": year_index,
        "year_weight": year_weight,
        "month": month,
        "month_weight": month_weight,
        "day": day,
        "day_weight": day_weight,
        "hour": hour_zhi + "时",
        "hour_name": hour_name,
        "hour_weight": hour_weight,
        "total_weight": round(total_weight, 1),
        "breakdown": f"{year_weight} + {month_weight} + {day_weight} + {hour_weight} = {round(total_weight, 1)}"
    }

# Calculate bone weight for lunar birthday
print("--- Yuan Tian Gang Bone Weight Calculation ---")
bone_weight_result = calculate_yuan_tian_gang_bone_weight(lunar_birthday)

print(f"\n农历生日: {bone_weight_result['lunar_date']}")
print(f"年份: {bone_weight_result['year']} (60年轮回索引: {bone_weight_result['year_index']})")
print(f"  年干支权重: {bone_weight_result['year_weight']}")
print(f"月份: {bone_weight_result['month']}")
print(f"  月份权重: {bone_weight_result['month_weight']}")
print(f"日期: {bone_weight_result['day']}")
print(f"  日期权重: {bone_weight_result['day_weight']}")
print(f"时辰: {bone_weight_result['hour']} ({bone_weight_result['hour_name']})")
print(f"  时辰权重: {bone_weight_result['hour_weight']}")
print(f"\n骨重计算: {bone_weight_result['breakdown']}")
print(f"=======================")
print(f"总骨重: {bone_weight_result['total_weight']} 两")

--- Yuan Tian Gang Bone Weight Calculation ---

农历生日: 一九八五年十月十四
年份: 1985 (60年轮回索引: 1)
  年干支权重: 0.9
月份: 10
  月份权重: 0.8
日期: 14
  日期权重: 1.7
时辰: 申时 (Shen)
  时辰权重: 0.8

骨重计算: 0.9 + 0.8 + 1.7 + 0.8 = 4.2
总骨重: 4.2 两


The "Golden Rule" of BaZi
There is a famous saying in Chinese metaphysics:

一命、二运、三风水、四积阴德、五读书

Destiny (BaZi)

Luck (The Cycles)

Feng Shui (Environment)

Character/Karma (Good Deeds)

Education (Self-Awareness)

A "perfect" BaZi is useless if the Luck Cycles (运) are terrible. Conversely, a "mediocre" BaZi can become legendary if the person enters a 20-year Golden Luck cycle and has the Education to take advantage of it.

Why Your "Imperfections" are Your Strengths
If you had a "perfect" chart, you would have no drive.

Conflict (Clashes): Often produce the greatest innovators and leaders because they are forced to change and grow.

Missing Elements: Often create the "hunger" that leads to massive wealth or achievement.

In [38]:
print('节气表 (Solar Terms):')  # Print header for solar terms
print(jieQi_table)

节气表 (Solar Terms):
{'DA_XUE': <lunar_python.Solar.Solar object at 0x000002398C9CEB10>, '冬至': <lunar_python.Solar.Solar object at 0x000002398C9CCD10>, '小寒': <lunar_python.Solar.Solar object at 0x000002398C9CF800>, '大寒': <lunar_python.Solar.Solar object at 0x000002398C9CC740>, '立春': <lunar_python.Solar.Solar object at 0x000002398C9CE2A0>, '雨水': <lunar_python.Solar.Solar object at 0x000002398C9CC2F0>, '惊蛰': <lunar_python.Solar.Solar object at 0x000002398C9CC0B0>, '春分': <lunar_python.Solar.Solar object at 0x000002398C9CFC80>, '清明': <lunar_python.Solar.Solar object at 0x000002398C9CEC60>, '谷雨': <lunar_python.Solar.Solar object at 0x000002398C9CF140>, '立夏': <lunar_python.Solar.Solar object at 0x000002398C9CD760>, '小满': <lunar_python.Solar.Solar object at 0x000002398C9CF710>, '芒种': <lunar_python.Solar.Solar object at 0x000002398C9CF7D0>, '夏至': <lunar_python.Solar.Solar object at 0x000002398CCBD640>, '小暑': <lunar_python.Solar.Solar object at 0x000002398CCBE750>, '大暑': <lunar_python.Solar.Solar

Strange. Mix of han yu pin yin and chinese chracters

In [39]:
import re
# 1. Get the raw dictionary with mixed keys
jieQiTable = lunar.getJieQiTable()

# 2. Use getJieQiList() to filter only the 24 standard Chinese names
# This removes 'DA_XUE', 'DONG_ZHI', etc., and keeps the order correct.
clean_jie_qi = {}
for key in jieQiTable.keys():
    # This regex matches the Unicode range for Chinese characters
    if re.search(r'[\u4e00-\u9fff]', str(key)):
        clean_jie_qi[key] = jieQiTable[key].toYmdHms()

# 3. Sort them by date so they make sense for the LLM
sorted_jie_qi = dict(sorted(clean_jie_qi.items(), key=lambda item: item[1]))

# 4. Print for your record or feed to the LLM
for name, time in sorted_jie_qi.items():
    print(f"{name}: {time}")

冬至: 1986-12-22 12:02:07
小寒: 1987-01-06 05:13:00
大寒: 1987-01-20 22:40:23
立春: 1987-02-04 16:51:40
雨水: 1987-02-19 12:49:57
惊蛰: 1987-03-06 10:53:37
春分: 1987-03-21 11:51:58
清明: 1987-04-05 15:44:08
谷雨: 1987-04-20 22:57:32
立夏: 1987-05-06 09:05:35
小满: 1987-05-21 22:10:01
芒种: 1987-06-06 13:18:58
夏至: 1987-06-22 06:10:45
小暑: 1987-07-07 23:38:39
大暑: 1987-07-23 17:06:02
立秋: 1987-08-08 09:29:13
处暑: 1987-08-24 00:09:50
白露: 1987-09-08 12:24:07
秋分: 1987-09-23 21:45:16
寒露: 1987-10-09 03:59:40
霜降: 1987-10-24 07:00:52
立冬: 1987-11-08 07:05:40
小雪: 1987-11-23 04:29:23
大雪: 1987-12-07 23:52:12


In [40]:
from lunar_python.util import LunarUtil

daYunArr = yun.getDaYun()

for i in range(1, len(daYunArr)):
    dy = daYunArr[i]
    gan_zhi = dy.getGanZhi()
    gan = gan_zhi[0]
    zhi = gan_zhi[1]

    # 1. Calculate Stem Ten God
    gan_shishen = LunarUtil.SHI_SHEN.get(baZi.getDayGan() + gan)

    # 2. Calculate Hidden Branch Ten Gods
    hide_gan = LunarUtil.ZHI_HIDE_GAN.get(zhi)
    zhi_shishen_list = [LunarUtil.SHI_SHEN.get(baZi.getDayGan() + h_gan) for h_gan in hide_gan]

    # 3. Format with Professional 3-Tier Labels
    # We map the results to: [本气 (Main), 中气 (Mid), 余气 (Residual)]
    labels = ["本气 (Main Qi)", "中气 (Mid Qi)", "余气 (Residual Qi)"]
    formatted_parts = []

    for idx, shishen in enumerate(zhi_shishen_list):
        if idx < len(labels):
            formatted_parts.append(f"{labels[idx]}:{shishen}")

    branch_info = " | ".join(formatted_parts)

    print(f"大运[{i}] {dy.getStartYear()}年 | {gan_zhi} | 天干十神: {gan_shishen}")
    print(f"       地支藏干: [{branch_info}]")


大运[1] 1989年 | 戊子 | 天干十神: 比肩
       地支藏干: [本气 (Main Qi):正财]
大运[2] 1999年 | 己丑 | 天干十神: 劫财
       地支藏干: [本气 (Main Qi):劫财 | 中气 (Mid Qi):正财 | 余气 (Residual Qi):伤官]
大运[3] 2009年 | 庚寅 | 天干十神: 食神
       地支藏干: [本气 (Main Qi):七杀 | 中气 (Mid Qi):偏印 | 余气 (Residual Qi):比肩]
大运[4] 2019年 | 辛卯 | 天干十神: 伤官
       地支藏干: [本气 (Main Qi):正官]
大运[5] 2029年 | 壬辰 | 天干十神: 偏财
       地支藏干: [本气 (Main Qi):比肩 | 中气 (Mid Qi):正官 | 余气 (Residual Qi):正财]
大运[6] 2039年 | 癸巳 | 天干十神: 正财
       地支藏干: [本气 (Main Qi):偏印 | 中气 (Mid Qi):食神 | 余气 (Residual Qi):比肩]
大运[7] 2049年 | 甲午 | 天干十神: 七杀
       地支藏干: [本气 (Main Qi):正印 | 中气 (Mid Qi):劫财]
大运[8] 2059年 | 乙未 | 天干十神: 正官
       地支藏干: [本气 (Main Qi):劫财 | 中气 (Mid Qi):正印 | 余气 (Residual Qi):正官]
大运[9] 2069年 | 丙申 | 天干十神: 偏印
       地支藏干: [本气 (Main Qi):食神 | 中气 (Mid Qi):偏财 | 余气 (Residual Qi):比肩]


In [41]:
from lunar_python.util import LunarUtil

# Choose a specific Luck Pillar (e.g., your current one)
# Assuming daYunArr[4] is your current 2019-2029 pillar
current_da_yun = daYunArr[4]
liuNianArr = current_da_yun.getLiuNian()

print(f"--- {current_da_yun.getGanZhi()} 大运中的流年分析 (Current Window) ---")

for ln in liuNianArr:
    # Filter for relevant years (e.g., 2024 to 2028)
    if 2025 <= ln.getYear() <= 2029:
        gz = ln.getGanZhi()
        gan, zhi = gz[0], gz[1]

        # 1. Stem Ten God
        gan_ss = LunarUtil.SHI_SHEN.get(baZi.getDayGan() + gan)

        # 2. Branch Hidden Ten Gods (Main/Mid/Res)
        h_gans = LunarUtil.ZHI_HIDE_GAN.get(zhi)
        ss_list = [LunarUtil.SHI_SHEN.get(baZi.getDayGan() + h) for h in h_gans]

        labels = ["本气(Main)", "中气(Mid)", "余气(Res)"]
        branch_ss = " | ".join([f"{labels[i]}:{ss_list[i]}" for i in range(len(ss_list))])

        print(f"流年 {ln.getYear()}年 ({ln.getAge()}岁) | {gz} | 天干:{gan_ss}")
        print(f"       地支藏干: [{branch_ss}]")


--- 辛卯 大运中的流年分析 (Current Window) ---
流年 2025年 (41岁) | 乙巳 | 天干:正官
       地支藏干: [本气(Main):偏印 | 中气(Mid):食神 | 余气(Res):比肩]
流年 2026年 (42岁) | 丙午 | 天干:偏印
       地支藏干: [本气(Main):正印 | 中气(Mid):劫财]
流年 2027年 (43岁) | 丁未 | 天干:正印
       地支藏干: [本气(Main):劫财 | 中气(Mid):正印 | 余气(Res):正官]
流年 2028年 (44岁) | 戊申 | 天干:比肩
       地支藏干: [本气(Main):食神 | 中气(Mid):偏财 | 余气(Res):比肩]


In [42]:
# 打印阴历 Print lunar date
print(lunar_birthday.toFullString())  # Print complete lunar date with zodiac information
print('')
# 阴历转阳历并打印 Convert lunar to solar and print
print(lunar_birthday.getSolar().toFullString())  # Convert lunar date to solar calendar and print full details

一九八五年十月十四 乙丑(牛)年 丁亥(猪)月 戊辰(龙)日 申(猴)时 纳音[海中金 屋上土 大林木 石榴木] 星期一 西方白虎 星宿[毕月乌](吉) 彭祖百忌[戊不受田田主不祥 辰不哭泣必主重丧] 喜神方位[巽](东南) 阳贵神方位[艮](东北) 阴贵神方位[坤](西南) 福神方位[艮](东北) 财神方位[坎](正北) 冲[(壬戌)狗] 煞[南]

1985-11-25 16:14:36 星期一 射手座


In [43]:
fu = lunar_birthday.getFu()
print(fu)
print(f"三伏 (Dog Days): {fu if fu else '非伏天 (Not in Dog Days)'}")

None
三伏 (Dog Days): 非伏天 (Not in Dog Days)


In [44]:
# Extracting the Precise Mathematical Coordinates
print("--- 命盘坐标 (Mathematical Coordinates) ---")
print(f"年干支索引: {lunar_birthday.getYearGanIndexExact()}, {lunar_birthday.getYearZhiIndexExact()}")
print(f"月干支索引: {lunar_birthday.getMonthGanIndexExact()}, {lunar_birthday.getMonthZhiIndexExact()}")
print(f"日干支索引: {lunar_birthday.getDayGanIndexExact2()}, {lunar_birthday.getDayZhiIndexExact2()}") # Sect 2
print(f"时干支索引: {lunar_birthday.getTimeGanIndex()}, {lunar_birthday.getTimeZhiIndex()}")


--- 命盘坐标 (Mathematical Coordinates) ---
年干支索引: 1, 1
月干支索引: 3, 11
日干支索引: 4, 4
时干支索引: 6, 8


In [45]:
# Limit the search to a specific range (e.g., 1920 to 2030)
# This prevents the LLM from getting confused by 18th-century dates.
results = Solar.fromBaZi("己酉", "癸酉", "甲辰", "丙寅", 0, 2025)

for d in results:
    if d.getYear() < 2050:
        print(f"Match Found: {d.toYmd()}")  # Print matching date

In [46]:
results = Solar.fromBaZi("己酉", "癸酉", "甲辰", "丙寅")

for d in results:
    lunar = d.getLunar()
    print(f"Auspicious Date: {d.toFullString()}")

    # In most versions of the library, it is getYearNineStar, not getNineStar
    # We use '3' as the sect for the 'Exact' calculation we discussed earlier
    year_star = lunar.getYearNineStar(3).toFullString()
    month_star = lunar.getMonthNineStar(3).toFullString()

    print(f"Energy Signature (Year): {year_star}")
    print(f"Energy Signature (Month): {month_star}")

Auspicious Date: 1969-09-26 04:00:00 星期五 天秤座
Energy Signature (Year): 四绿木 巽(东南) 天权 玄空[文曲 吉] 奇门[天辅 大吉 杜门 阳] 太乙[招摇 安神]
Energy Signature (Month): 一白水 坎(正北) 天枢 玄空[贪狼 吉] 奇门[天蓬 大凶 休门 阳] 太乙[太乙 吉神]


In [47]:
def get_comprehensive_shen_sha(lunar_birthday):
    baZi = lunar_birthday.getEightChar()

    # 1. Basic Data Extraction
    yg, yz = baZi.getYearGan(), baZi.getYearZhi()
    mg, mz = baZi.getMonthGan(), baZi.getMonthZhi()
    dg, dz = baZi.getDayGan(), baZi.getDayZhi()
    tg, tz = baZi.getTimeGan(), baZi.getTimeZhi()

    # Organize for iteration
    stems = [yg, mg, dg, tg]
    branches = [yz, mz, dz, tz]
    pillar_names = ['Year', 'Month', 'Day', 'Hour']
    results = {name: [] for name in pillar_names}

    # --- 1. Taiji Nobleman (太极贵人) ---
    # Based on Day Stem: For 戊 (Earth), it looks for all Earth branches
    taiji_map = {
        '甲':['子','午'], '乙':['子','午'], '丙':['卯','酉'], '丁':['卯','酉'],
        '戊':['辰','戌','丑','未'], '己':['辰','戌','丑','未'],
        '庚':['寅','亥'], '辛':['寅','亥'], '壬':['申','巳'], '癸':['申','巳']
    }
    for i, zhi in enumerate(branches):
        if zhi in taiji_map.get(dg, []):
            results[pillar_names[i]].append("Taiji (太极贵人)")

    # --- 2. Red Charm / Hong Yan (红艳) ---
    # Based on Day Stem: For 戊, it is 辰 (Dragon)
    red_charm_map = {'甲':'午', '乙':'申', '丙':'寅', '丁':'未', '戊':'辰', '己':'辰', '庚':'戌', '辛':'酉', '壬':'子', '癸':'申'}
    if dz == red_charm_map.get(dg):
        results['Day'].append("Red Charm (红艳)")

    # --- 3. Imperial Seal (国印贵人) ---
    # Based on Day Stem: For 戊, it is 丑 (Ox)
    imperial_map = {'甲':'戌', '乙':'亥', '丙':'丑', '丁':'寅', '戊':'丑', '己':'寅', '庚':'辰', '辛':'巳', '壬':'未', '癸':'申'}
    for i, zhi in enumerate(branches):
        if zhi == imperial_map.get(dg):
            results[pillar_names[i]].append("Imperial Seal (国印)")

    # --- 4. Virtuous & Talented (德秀贵人) ---
    # This is complex: Based on the Month Branch and the Stems present in the chart
    # For a 亥 (Pig) month, you need 丁 (Yin Fire) and 壬 (Yang Water)
    if mz in ['亥', '卯', '未']:
        if '丁' in stems and '壬' in stems:
            results['Year'].append("Virtuous and Talented Noble (德秀贵人)")

    # --- 2. VIRTUE STARS (Month Branch Based) ---
    # Heavenly Virtue (天德贵人)
    tv_map = {'子':'巳', '丑':'庚', '寅':'丁', '卯':'申', '辰':'壬', '巳':'辛', '午':'亥', '未':'甲', '申':'癸', '酉':'寅', '戌':'丙', '亥':'乙'}
    target_tv = tv_map.get(mz)
    # Virtue Union (天德合)
    vu_map = {'子':'申', '丑':'乙', '寅':'壬', '卯':'己', '辰':'丁', '巳':'丙', '午':'寅', '未':'己', '申':'巳', '酉':'庚', '戌':'辛', '亥':'庚'}
    target_vu = vu_map.get(mz)

    for i in range(4):
        if stems[i] == target_tv or branches[i] == target_tv:
            results[pillar_names[i]].append("Heavenly Virtue (天德贵人)")
        if stems[i] == target_vu or branches[i] == target_vu:
            results[pillar_names[i]].append("Virtue Union (天德合)")

    # --- 3. DAY STEM BASED STARS ---
    # Fortune Star (福星贵人)
    fortune_map = {'甲':'子', '乙':'丑', '丙':'子', '丁':'亥', '戊':'申', '己':'未', '庚':'午', '辛':'巳', '壬':'辰', '癸':'卯'}
    # Heavenly Chef (天厨贵人)
    chef_map = {'甲':'巳', '乙':'午', '丙':'巳', '丁':'午', '戊':'申', '己':'酉', '庚':'亥', '辛':'子', '壬':'寅', '癸':'卯'}
    # Heavenly Noble (天官贵人)
    official_map = {'甲':'未', '乙':'辰', '丙':'巳', '丁':'亥', '戊':'卯', '己':'申', '庚':'丑', '辛':'午', '壬':'寅', '癸':'酉'}

    for i, zhi in enumerate(branches):
        if zhi == fortune_map.get(dg):
            results[pillar_names[i]].append("Fortune Star (福星贵人)")
        if zhi == chef_map.get(dg):
            results[pillar_names[i]].append("Heavenly Chef (天厨贵人)")
        if zhi == official_map.get(dg):
            results[pillar_names[i]].append("Heavenly Noble (天官贵人)")

    # --- 4. THE "DEMONS" (SHA) ---
    # Blood Blade (血刃) - Based on Month Branch
    blood_map = {'子':'戌', '丑':'酉', '寅':'申', '卯':'未', '辰':'午', '巳':'巳', '午':'辰', '未':'卯', '申':'寅', '酉':'丑', '戌':'子', '亥':'亥'}
    # Lost Spirit (亡神) - Based on Year/Day Branch
    lost_map = {'申':'亥', '子':'亥', '辰':'亥', '寅':'巳', '午':'巳', '戌':'巳', '巳':'申', '酉':'申', '丑':'申', '亥':'寅', '卯':'寅', '未':'寅'}

    for i, zhi in enumerate(branches):
        if zhi == blood_map.get(mz):
            results[pillar_names[i]].append("Blood Blade (血刃)")
        if zhi in [lost_map.get(yz), lost_map.get(dz)]:
            results[pillar_names[i]].append("Lost Spirit (亡神)")

    return results

stars = get_comprehensive_shen_sha(lunar_birthday)
print(f"Comprehensive Shen Sha Analysis: {stars}")

Comprehensive Shen Sha Analysis: {'Year': ['Taiji (太极贵人)', 'Imperial Seal (国印)', 'Heavenly Virtue (天德贵人)'], 'Month': ['Blood Blade (血刃)', 'Lost Spirit (亡神)'], 'Day': ['Taiji (太极贵人)', 'Red Charm (红艳)'], 'Hour': ['Virtue Union (天德合)', 'Fortune Star (福星贵人)', 'Heavenly Chef (天厨贵人)', 'Lost Spirit (亡神)']}


In [48]:
from lunar_python import Solar, Lunar

# --- DATA DICTIONARIES (Provided by you) ---
year_shens = {
    '孤辰': {"子":"寅", "丑":"寅", "寅":"巳", "卯":"巳", "辰":"巳", "巳":"申", "午":"申", "未":"申", "申":"亥", "酉":"亥", "戌":"亥", "亥":"寅"},
    '寡宿': {"子":"戌", "丑":"戌", "寅":"丑", "卯":"丑", "辰":"丑", "巳":"辰", "午":"辰", "未":"辰", "申":"未", "酉":"未", "戌":"未", "亥":"戌"},
    '大耗': {"子":"巳未", "丑":"午申", "寅":"未酉", "卯":"申戌", "辰":"酉亥", "巳":"戌子", "午":"亥丑", "未":"子寅", "申":"丑卯", "酉":"寅辰", "戌":"卯巳", "亥":"辰午"},
}

month_shens = {
    '天德': {"子":"巳", "丑":"庚", "寅":"丁", "卯":"申", "辰":"壬", "巳":"辛", "午":"亥", "未":"甲", "申":"癸", "酉":"寅", "戌":"丙", "亥":"乙"},
    '月德': {"子":"壬", "丑":"庚", "寅":"丙", "卯":"甲", "辰":"壬", "巳":"庚", "午":"丙", "未":"甲", "申":"壬", "酉":"庚", "戌":"丙", "亥":"甲"},
}

day_shens = {
    '将星': {"子":"子", "丑":"酉", "寅":"午", "卯":"卯", "辰":"子", "巳":"酉", "午":"午", "未":"卯", "申":"子", "酉":"酉", "戌":"午", "亥":"卯"},
    '华盖': {"子":"辰", "丑":"丑", "寅":"戌", "卯":"未", "辰":"辰", "巳":"丑", "午":"戌", "未":"未", "申":"辰", "酉":"丑", "戌":"戌", "亥":"未"},
    '驿马': {"子":"寅", "丑":"亥", "寅":"申", "卯":"巳", "辰":"寅", "巳":"亥", "午":"申", "未":"巳", "申":"寅", "酉":"亥", "戌":"申", "亥":"巳"},
    '劫煞': {"子":"巳", "丑":"寅", "寅":"亥", "卯":"申", "辰":"巳", "巳":"寅", "午":"亥", "未":"申", "申":"巳", "酉":"寅", "戌":"亥", "亥":"申"},
    '亡神': {"子":"亥", "丑":"申", "寅":"巳", "卯":"寅", "辰":"亥", "巳":"申", "午":"巳", "未":"寅", "申":"亥", "酉":"申", "戌":"巳", "亥":"寅"},
    '桃花': {"子":"酉", "丑":"午", "寅":"卯", "卯":"子", "辰":"酉", "巳":"午", "午":"卯", "未":"子", "申":"酉", "酉":"午", "戌":"卯", "亥":"子"},
}

g_shens = {
    '天乙': {"甲":'未丑', "乙":"申子", "丙":"酉亥", "丁":"酉亥", "戊":'未丑', "己":"申子", "庚": "未丑", "辛":"寅午", "壬": "卯巳", "癸":"卯巳"},
    '文昌': {"甲":'巳', "乙":"午", "丙":"申", "丁":"酉", "戊":"申", "己":"酉", "庚": "亥", "辛":"子", "壬": "寅", "癸":"丑"},
    '阳刃': {"甲":'卯', "乙":"", "丙":"午", "丁":"", "戊":"午", "己":"", "庚": "酉", "辛":"", "壬": "子", "癸":""},
    '红艳': {"甲":'午', "乙":"午", "丙":"寅", "丁":"未", "戊":"辰", "己":"辰", "庚": "戌", "辛":"酉", "壬": "子", "癸":"申"},
}

shens_infos = {
    '孤辰': "孤僻、孤独：月支容易不合群、容易30岁以后才结婚。女命官杀月干坐顾辰、独居概率大，时支则有阴道之心。",
    '寡宿': "类似孤辰，同柱有天月德没关系。男怕孤，女怕寡。",
    '大耗': "意外破损，单独没关系。与桃花或驿马之类同柱则危险。",
    '天德': "先天有福，日干终生有福。忌讳冲克，不怕合。女命与夫星同干更佳。",
    '月德': "先天有福，日干终生有福。忌讳冲克，不怕合。女命与夫星同干更佳。",
    '将星': "有理想、气度、即从容不迫。",
    '华盖': "有艺术、水准与命格相关。",
    '驿马': "多迁移、水准与命格相关。女驿马合贵人，终沦落风尘。",
    '劫煞': "与贵人同柱没关系、与亡神对冲。会三刑不佳，其他情况还好。为日主所克无大碍。",
    '亡神': "与贵人同柱没关系、与劫煞对冲。会三刑不佳，其他情况还好。为日主所克无大碍。",
    '桃花': "凶居多、女正官坐桃花吉。",
    '天乙': "后天解难、女命不适合多",
    '文昌': "诗书佳，未必有福，女命多参考李清照。",
    '阳刃': "性格刚强，女命未必佳。",
    '红艳': "爱得执著，不顾及地位差异。",
    '天乙': "天乙贵人：众神之首，遇难呈祥。代表一生多得贵人提拔，能化解凶灾。",
    '天德': "天德贵人：象征心地善良，逢凶化吉，一生少有官非血光之灾。",
    '月德': "月德贵人：与天德功用类似，象征先天的福分与保护力。",
    '将星': "将星：代表领导才能、组织能力，具有大将之风，处事沉稳。",
    '金舆': "金舆贵人：主配偶助力，或能因婚得财；代表举止端庄，生活优渥。",
    '文昌': "文昌贵人：主聪明过人，学习力强，利于学业、考试及文书工作。",
    '华盖': "华盖：艺术与哲学之星。代表才华横溢但性情孤僻，与佛道宗教有缘。",
    '驿马': "驿马：主变动、远行、奔波。利于事业扩张、迁移或出差。",
    '阳刃': "阳刃（羊刃）：代表极端、权力与刚毅。若无制约则易有血光或手术之灾。",
    '劫煞': "劫煞：主争端、破财、官非，常感不安与焦虑。",
    '空亡': "空亡：象征虚幻、徒劳。吉神遇之福力减半，凶神遇之凶力减小。"
}

# --- CORE ENGINE ---

def get_full_analysis(lunar_birthday):

    baZi = lunar_birthday.getEightChar()

    # Extract Gans (Stems) and Zhis (Branches)
    gans = [baZi.getYearGan(), baZi.getMonthGan(), baZi.getDayGan(), baZi.getTimeGan()]
    zhis = [baZi.getYearZhi(), baZi.getMonthZhi(), baZi.getDayZhi(), baZi.getTimeZhi()]

    me = gans[2] # Day Stem
    day_voids = baZi.getDayXunKong() # Returns two branches, e.g., "戌亥"

    # Container for results (0: Year, 1: Month, 2: Day, 3: Hour)
    strs = ['', '', '', '']
    all_found_shens = []

    # 1. Year Branch Based
    for item, mapping in year_shens.items():
        for i in (1, 2, 3):
            if zhis[i] in mapping.get(zhis[0], ""):
                strs[i] = item if not strs[i] else f"{strs[i]}  {item}"
                all_found_shens.append(item)

    # 2. Month Branch Based
    for item, mapping in month_shens.items():
        for i in range(4):
            target = mapping.get(zhis[1], "")
            if gans[i] in target or zhis[i] in target:
                strs[i] = item if not strs[i] else f"{strs[i]}  {item}"
                all_found_shens.append(item)

    # 3. Day Branch Based
    for item, mapping in day_shens.items():
        for i in (0, 1, 3):
            if zhis[i] in mapping.get(zhis[2], ""):
                strs[i] = item if not strs[i] else f"{strs[i]}  {item}"
                all_found_shens.append(item)

    # 4. Day Stem Based (Me)
    for item, mapping in g_shens.items():
        for i in range(4):
            if zhis[i] in mapping.get(me, ""):
                strs[i] = item if not strs[i] else f"{strs[i]}  {item}"
                all_found_shens.append(item)

    # 5. Add Void (空亡) markers
    for i in range(4):
        if zhis[i] in day_voids:
            strs[i] = f"{strs[i]} [空]".strip()

    return strs, list(set(all_found_shens)), gans, zhis

# --- EXECUTION ---

if __name__ == "__main__":

    results, unique_shens, gans, zhis = get_full_analysis(lunar_birthday)
    print("--- Comprehensive Shen Sha Analysis ---")
    pillar_names = ["Year ", "Month", "Day  ", "Hour "]
    for i in range(4):
        print(f"{pillar_names[i]} Pillar: {gans[i]}{zhis[i]} | Stars: {results[i]}")

    print("\n--- Shen Sha Meanings ---")
    for s in sorted(unique_shens):
        print(f"【{s}】: {shens_infos.get(s, 'Description missing.')}")

--- Comprehensive Shen Sha Analysis ---
Year  Pillar: 乙丑 | Stars: 天德  天乙
Month Pillar: 丁亥 | Stars: 亡神 [空]
Day   Pillar: 戊辰 | Stars: 红艳
Hour  Pillar: 庚申 | Stars: 大耗  文昌

--- Shen Sha Meanings ---
【亡神】: 与贵人同柱没关系、与劫煞对冲。会三刑不佳，其他情况还好。为日主所克无大碍。
【大耗】: 意外破损，单独没关系。与桃花或驿马之类同柱则危险。
【天乙】: 天乙贵人：众神之首，遇难呈祥。代表一生多得贵人提拔，能化解凶灾。
【天德】: 天德贵人：象征心地善良，逢凶化吉，一生少有官非血光之灾。
【文昌】: 文昌贵人：主聪明过人，学习力强，利于学业、考试及文书工作。
【红艳】: 爱得执著，不顾及地位差异。


In [49]:
from lunar_python import Solar, Lunar

# --- EXTENDED DATA DICTIONARIES ---

year_shens = {
    '孤辰': {"子":"寅", "丑":"寅", "寅":"巳", "卯":"巳", "辰":"巳", "巳":"申", "午":"申", "未":"申", "申":"亥", "酉":"亥", "戌":"亥", "亥":"寅"},
    '寡宿': {"子":"戌", "丑":"戌", "寅":"丑", "卯":"丑", "辰":"丑", "巳":"辰", "午":"辰", "未":"辰", "申":"未", "酉":"未", "戌":"未", "亥":"戌"},
    '大耗': {"子":"巳未", "丑":"午申", "寅":"未酉", "卯":"申戌", "辰":"酉亥", "巳":"戌子", "午":"亥丑", "未":"子寅", "申":"丑卯", "酉":"寅辰", "戌":"卯巳", "亥":"辰午"},
    '吊客': {"子":"戌", "丑":"亥", "寅":"子", "卯":"丑", "辰":"寅", "巳":"卯", "午":"辰", "未":"巳", "申":"午", "酉":"未", "戌":"申", "亥":"酉"},
    '丧门': {"子":"寅", "丑":"卯", "寅":"辰", "卯":"巳", "辰":"午", "巳":"未", "午":"申", "未":"酉", "申":"戌", "酉":"亥", "戌":"子", "亥":"丑"},
    '白虎': {"子":"申", "丑":"酉", "寅":"戌", "卯":"亥", "辰":"子", "巳":"丑", "午":"寅", "未":"卯", "申":"辰", "酉":"巳", "戌":"午", "亥":"未"},
}

month_shens = {
    '天德': {"子":"巳", "丑":"庚", "寅":"丁", "卯":"申", "辰":"壬", "巳":"辛", "午":"亥", "未":"甲", "申":"癸", "酉":"寅", "戌":"丙", "亥":"乙"},
    '月德': {"子":"壬", "丑":"庚", "寅":"丙", "卯":"甲", "辰":"壬", "巳":"庚", "午":"丙", "未":"甲", "申":"壬", "酉":"庚", "戌":"丙", "亥":"甲"},
    '血刃': {"子":"戌", "丑":"酉", "寅":"申", "卯":"未", "辰":"午", "巳":"巳", "午":"辰", "未":"卯", "申":"寅", "酉":"丑", "戌":"子", "亥":"亥"},
    '天医': {"子":"亥", "丑":"子", "寅":"丑", "卯":"寅", "辰":"卯", "巳":"辰", "午":"巳", "未":"午", "申":"未", "酉":"申", "戌":"酉", "亥":"戌"},
    '天喜': {"子":"酉", "丑":"申", "寅":"未", "卯":"午", "辰":"巳", "巳":"辰", "午":"卯", "未":"寅", "申":"丑", "酉":"子", "戌":"亥", "亥":"戌"},
}

day_shens = {
    '将星': {"子":"子", "丑":"酉", "寅":"午", "卯":"卯", "辰":"子", "巳":"酉", "午":"午", "未":"卯", "申":"子", "酉":"酉", "戌":"午", "亥":"卯"},
    '华盖': {"子":"辰", "丑":"丑", "寅":"戌", "卯":"未", "辰":"辰", "巳":"丑", "午":"戌", "未":"未", "申":"辰", "酉":"丑", "戌":"戌", "亥":"未"},
    '驿马': {"子":"寅", "丑":"亥", "寅":"申", "卯":"巳", "辰":"寅", "巳":"亥", "午":"申", "未":"巳", "申":"寅", "酉":"亥", "戌":"申", "亥":"巳"},
    '劫煞': {"子":"巳", "丑":"寅", "寅":"亥", "卯":"申", "辰":"巳", "巳":"寅", "午":"亥", "未":"申", "申":"巳", "酉":"寅", "戌":"亥", "亥":"申"},
    '亡神': {"子":"亥", "丑":"申", "寅":"巳", "卯":"寅", "辰":"亥", "巳":"申", "午":"巳", "未":"寅", "申":"亥", "酉":"申", "戌":"巳", "亥":"寅"},
    '桃花': {"子":"酉", "丑":"午", "寅":"卯", "卯":"子", "辰":"酉", "巳":"午", "午":"卯", "未":"子", "申":"酉", "酉":"午", "戌":"卯", "亥":"子"},
}

g_shens = {
    '天乙': {"甲":'未丑', "乙":"申子", "丙":"酉亥", "丁":"酉亥", "戊":'未丑', "己":"申子", "庚": "未丑", "辛":"寅午", "壬": "卯巳", "癸":"卯巳"},
    '文昌': {"甲":'巳', "乙":"午", "丙":"申", "丁":"酉", "戊":"申", "己":"酉", "庚": "亥", "辛":"子", "壬": "寅", "癸":"丑"},
    '阳刃': {"甲":'卯', "丙":"午", "戊":"午", "庚": "酉", "壬": "子"},
    '红艳': {"甲":'午', "乙":"午", "丙":"寅", "丁":"未", "戊":"辰", "己":"辰", "庚": "戌", "辛":"酉", "壬": "子", "癸":"申"},
    '金舆': {"甲":'辰', "乙":"巳", "丙":"未", "丁":"申", "戊":"未", "己":"申", "庚":"戌", "辛":"亥", "壬":"丑", "癸":"寅"},
    '国印': {"甲":'戌', "乙":"亥", "丙":"丑", "丁":"寅", "戊":"丑", "己":"寅", "庚":"辰", "辛":"巳", "壬":"未", "癸":"申"},
    '太极': {"甲":"子午", "乙":"子午", "丙":"卯酉", "丁":"卯酉", "戊":"辰戌丑未", "己":"辰戌丑未", "庚":"寅亥", "辛":"寅亥", "壬":"巳申", "癸":"巳申"},
    '福星': {"甲":"寅子", "乙":"亥丑", "丙":"戌", "丁":"酉", "戊":"申", "己":"未", "庚":"午", "辛":"巳", "壬":"辰", "癸":"卯"},
    '天厨': {"甲":"巳", "乙":"午", "丙":"巳", "丁":"午", "戊":"申", "己":"酉", "庚":"亥", "辛":"子", "壬":"寅", "癸":"丑"},
    '禄神': {"甲":"寅", "乙":"卯", "丙":"巳", "丁":"午", "戊":"巳", "己":"午", "庚":"申", "辛":"酉", "壬":"亥", "癸":"子"},
    '词馆': {"甲":"庚寅", "乙":"辛卯", "丙":"乙巳", "丁":"甲午", "戊":"乙巳", "己":"甲午", "庚":"壬申", "辛":"癸酉", "壬":"丁亥", "癸":"丙子"},
}

# --- CORE ENGINE ---

def get_full_analysis(lunar_birthday):
    baZi = lunar_birthday.getEightChar()

    gans = [baZi.getYearGan(), baZi.getMonthGan(), baZi.getDayGan(), baZi.getTimeGan()]
    zhis = [baZi.getYearZhi(), baZi.getMonthZhi(), baZi.getDayZhi(), baZi.getTimeZhi()]

    me = gans[2] # Day Stem
    day_pillar = gans[2] + zhis[2]

    strs = ['', '', '', '']
    all_found_shens = []

    # 1. Year Branch Based (Comparing Year Branch to Month, Day, Hour)
    for item, mapping in year_shens.items():
        lookup = mapping.get(zhis[0], "")
        for i in (1, 2, 3):
            if zhis[i] in lookup:
                strs[i] += f" {item}"
                all_found_shens.append(item)

    # 2. Month Branch Based (Comparing Month Branch to all 4 pillars)
    for item, mapping in month_shens.items():
        lookup = mapping.get(zhis[1], "")
        for i in range(4):
            # Check Stems (for Virtues) or Branches (for Blood Blade/Joy)
            if gans[i] in lookup or zhis[i] in lookup:
                strs[i] += f" {item}"
                all_found_shens.append(item)

    # 3. Day Branch Based (Comparing Day Branch to Year, Month, Hour)
    for item, mapping in day_shens.items():
        lookup = mapping.get(zhis[2], "")
        for i in (0, 1, 3):
            if zhis[i] in lookup:
                strs[i] += f" {item}"
                all_found_shens.append(item)

    # 4. Day Stem Based (Comparing Day Stem to all 4 Branches/Pillars)
    for item, mapping in g_shens.items():
        lookup = mapping.get(me, "")
        for i in range(4):
            # Special check for '词馆' (Library) which checks full pillar strings
            if item == '词馆':
                if (gans[i] + zhis[i]) in lookup:
                    strs[i] += f" {item}"
                    all_found_shens.append(item)
            # Standard check for Branches
            elif zhis[i] in lookup:
                strs[i] += f" {item}"
                all_found_shens.append(item)

    # 5. SPECIAL: Kui Gang (Only checks the Day Pillar)
    if day_pillar in ["庚辰", "庚戌", "戊戌", "壬辰"]:
        strs[2] += " 魁罡"
        all_found_shens.append("魁罡")

    return [s.strip() for s in strs], list(set(all_found_shens)), gans, zhis

# --- EXECUTION ---

if __name__ == "__main__":
    # Test Date
    results, unique_shens, gans, zhis = get_full_analysis(lunar_birthday)

    pillar_names = ["Year", "Month", "Day", "Hour"]
    print(f"Eight Char: {' '.join(gans)} / {' '.join(zhis)}\n")

    for i in range(4):
        print(f"{pillar_names[i]}: {gans[i]}{zhis[i]} | Stars: {results[i]}")

Eight Char: 乙 丁 戊 庚 / 丑 亥 辰 申

Year: 乙丑 | Stars: 天德 天乙 国印 太极
Month: 丁亥 | Stars: 吊客 血刃 亡神
Day: 戊辰 | Stars: 红艳 太极
Hour: 庚申 | Stars: 大耗 文昌 福星 天厨


In [54]:
from lunar_python import Solar, Lunar

# --- SHEN SHA DICTIONARIES ---

year_shens = {
    # --- POSITIVE / NEUTRAL ---
    '红鸾': {"子":"卯", "丑":"寅", "寅":"丑", "卯":"子", "辰":"亥", "巳":"戌", "午":"酉", "未":"申", "申":"未", "酉":"午", "戌":"巳", "亥":"辰"}, # Red Phoenix: Primary star for marriage, romance, and natural charm.
    '天喜': {"子":"酉", "丑":"申", "寅":"未", "卯":"午", "辰":"巳", "巳":"辰", "午":"卯", "未":"寅", "申":"丑", "酉":"子", "戌":"亥", "亥":"戌"}, # Heavenly Joy: Brings happiness, celebration, and assists Hong Luan in romance.

    # --- CHALLENGING / SHA ---
    '孤辰': {"子":"寅", "丑":"寅", "寅":"巳", "卯":"巳", "辰":"巳", "巳":"申", "午":"申", "未":"申", "申":"亥", "酉":"亥", "戌":"亥", "亥":"寅"}, # Lonesome Star: Represents emotional isolation, independence, or feeling misunderstood.
    '寡宿': {"子":"戌", "丑":"戌", "寅":"丑", "卯":"丑", "辰":"丑", "巳":"辰", "午":"辰", "未":"辰", "申":"未", "酉":"未", "戌":"未", "亥":"戌"}, # Solitary Star: Affects relationships; suggests a preference for solitude or distance from a spouse.
    '大耗': {"子":"巳未", "丑":"午申", "寅":"未酉", "卯":"申戌", "辰":"酉亥", "巳":"戌子", "午":"亥丑", "未":"子寅", "申":"丑卯", "酉":"寅辰", "戌":"卯巳", "亥":"辰午"}, # Great Consumer: Indicator of wealth leakage, unexpected expenses, or financial instability.
    '吊客': {"子":"戌", "丑":"亥", "寅":"子", "卯":"丑", "辰":"寅", "巳":"卯", "午":"辰", "未":"巳", "申":"午", "酉":"未", "戌":"申", "亥":"酉"}, # Condolence Guest: Associated with mourning or visiting the sick; suggests cautiousness in family health.
    '丧门': {"子":"寅", "丑":"卯", "寅":"辰", "卯":"巳", "辰":"午", "巳":"未", "午":"申", "未":"酉", "申":"戌", "酉":"亥", "戌":"子", "亥":"丑"}, # Funeral Door: Similar to Diao Ke; indicates potential for bad news or low energy in specific years.
    '白虎': {"子":"申", "丑":"酉", "寅":"戌", "卯":"亥", "辰":"子", "巳":"丑", "午":"寅", "未":"卯", "申":"辰", "酉":"巳", "戌":"午", "亥":"未"}, # White Tiger: Represents hidden danger, sudden accidents, or a very sharp, aggressive personality.
    '元辰': {"子":"未", "丑":"申", "寅":"酉", "卯":"戌", "辰":"亥", "巳":"子", "午":"丑", "未":"寅", "申":"卯", "酉":"辰", "戌":"巳", "亥":"午"}, # Great Opposition: Represents a lack of clarity, poor coordination, or things not going as planned.
}

month_shens = {
    '天德': {"子":"巳", "丑":"庚", "寅":"丁", "卯":"申", "辰":"壬", "巳":"辛", "午":"亥", "未":"甲", "申":"癸", "酉":"寅", "戌":"丙", "亥":"乙"}, # Heavenly Virtue: One of the strongest protective stars; dissolves many negative Sha stars.
    '月德': {"子":"壬", "丑":"庚", "寅":"丙", "卯":"甲", "辰":"壬", "巳":"庚", "午":"丙", "未":"甲", "申":"壬", "酉":"庚", "戌":"丙", "亥":"甲"}, # Monthly Virtue: Brings peace, helps resolve conflicts, and attracts helpful people (Nobles).
    '血刃': {"子":"戌", "丑":"酉", "寅":"申", "卯":"未", "辰":"午", "巳":"巳", "午":"辰", "未":"卯", "申":"寅", "酉":"丑", "戌":"子", "亥":"亥"}, # Blood Blade: Indicates risk of minor physical injuries, surgery, or sharp-object accidents.
    '天医': {"子":"亥", "丑":"子", "寅":"丑", "卯":"寅", "辰":"卯", "巳":"辰", "午":"巳", "未":"午", "申":"未", "酉":"申", "戌":"酉", "亥":"戌"}, # Heavenly Physician: Good for health and longevity; also indicates a talent for medical or healing professions.
    '德秀': {"子":"壬癸戊己丙辛甲己", "丑":"庚辛乙庚", "寅":"丙丁戊癸", "卯":"甲乙丁壬", "辰":"壬癸戊己丙辛甲己", "巳":"庚辛乙庚", "午":"丙丁戊癸", "未":"甲乙丁壬", "申":"壬癸戊己丙辛甲己", "酉":"庚辛乙庚", "戌":"丙丁戊癸", "亥":"甲乙丁壬"}, # Virtue & Elegance: Represents refined character, intelligence, and a balanced, dignified personality.
}

day_earthly_branches_shens = {
    '将星': {"子":"子", "丑":"酉", "寅":"午", "卯":"卯", "辰":"子", "巳":"酉", "午":"午", "未":"卯", "申":"子", "酉":"酉", "戌":"午", "亥":"卯"}, # General Star: Leadership, authority, and the ability to command or organize others.
    '华盖': {"子":"辰", "丑":"丑", "寅":"戌", "卯":"未", "辰":"辰", "巳":"丑", "午":"戌", "未":"未", "申":"辰", "酉":"丑", "戌":"戌", "亥":"未"}, # Elegant Seal: Talent in arts/culture; can indicate a lonely but highly spiritual or intellectual path.
    '驿马': {"子":"寅", "丑":"亥", "寅":"申", "卯":"巳", "辰":"寅", "巳":"亥", "午":"申", "未":"巳", "申":"寅", "酉":"亥", "戌":"申", "亥":"巳"}, # Sky Horse: Movement, travel, career changes, and efficiency in getting things done.
    '劫煞': {"子":"巳", "丑":"寅", "寅":"亥", "卯":"申", "辰":"巳", "巳":"寅", "午":"亥", "未":"申", "申":"巳", "酉":"寅", "戌":"亥", "亥":"申"}, # Robbery Sha: Obstacles or loss caused by others; requires alertness in business and trust.
    '亡神': {"子":"亥", "丑":"申", "寅":"巳", "卯":"寅", "辰":"亥", "巳":"申", "午":"巳", "未":"寅", "申":"亥", "酉":"申", "戌":"巳", "亥":"寅"}, # Death God: Represents deep thinking and strategy, but can lead to legal issues or mental stress if negative.
    '桃花': {"子":"酉", "丑":"午", "寅":"卯", "卯":"子", "辰":"酉", "巳":"午", "午":"卯", "未":"子", "申":"酉", "酉":"午", "戌":"卯", "亥":"子"}, # Peach Blossom: Social magnetism, popularity, and attraction; can lead to romance or social scandals.
}

day_heavenly_stem_shens = {
    '天乙': {"甲":'未丑', "乙":"申子", "丙":"酉亥", "丁":"酉亥", "戊":'未丑', "己":"申子", "庚": "未丑", "辛":"寅午", "壬": "卯巳", "癸":"卯巳"}, # Heavenly Noble: The ultimate guardian angel star; turns bad luck into good through the help of others.
    '文昌': {"甲":'巳', "乙":"午", "丙":"申", "丁":"酉", "戊":"申", "己":"酉", "庚": "亥", "辛":"子", "壬": "寅", "癸":"丑"}, # Intelligence Star: High IQ, academic success, literary talent, and quick learning.
    '阳刃': {"甲":'卯', "丙":"午", "戊":"午", "庚": "酉", "壬": "子"}, # Goat Blade: Extreme willpower and bravery, but also indicates a fierce temper or risk of injury.
    '红艳': {"甲":'午', "乙":"午", "丙":"寅", "丁":"未", "戊":"辰", "己":"辰", "庚": "戌", "辛":"酉", "壬": "子", "癸":"申"}, # Red Beauty: Personal seductive charm; more intense and private than Peach Blossom.
    '金舆': {"甲":'辰', "乙":"巳", "丙":"未", "丁":"申", "戊":"未", "己":"申", "庚":"戌", "辛":"亥", "壬":"丑", "癸":"寅"}, # Golden Carriage: Wealth and social status; suggests a comfortable life and high-end transport.
    '国印': {"甲":'戌', "乙":"亥", "丙":"丑", "丁":"寅", "戊":"丑", "己":"寅", "庚":"辰", "辛":"巳", "壬":"未", "癸":"申"}, # National Seal: Authority, reliability, and official power; good for civil servants or managers.
    '太极': {"甲":"子午", "乙":"子午", "丙":"卯酉", "丁":"卯酉", "戊":"辰戌丑未", "己":"辰戌丑未", "庚":"寅亥", "辛":"寅亥", "壬":"巳申", "癸":"巳申"}, # Taiji Noble: Interest in spirituality, metaphysics, and deep wisdom.
    '福星': {"甲":"寅子", "乙":"亥丑", "丙":"戌", "丁":"酉", "戊":"申", "己":"未", "庚":"午", "辛":"巳", "壬":"辰", "癸":"卯"}, # Prosperity Star: General luck in daily life; ensures that basic needs and "luck of the draw" are favorable.
    '天厨': {"甲":"巳", "乙":"午", "丙":"巳", "丁":"午", "戊":"申", "己":"酉", "庚":"亥", "辛":"子", "壬":"寅", "癸":"丑"}, # Heavenly Kitchen: Luck in food and wealth; suggests a "well-fed" life and culinary talent.
    '禄神': {"甲":"寅", "乙":"卯", "丙":"巳", "丁":"午", "戊":"巳", "己":"午", "庚":"申", "辛":"酉", "壬":"亥", "癸":"子"}, # Prosperity Star/Lu: Representing the "Salary"; the ability to earn wealth and maintain a high standard of living.
    '词馆': {"甲":"庚寅", "乙":"辛卯", "丙":"乙巳", "丁":"甲午", "戊":"乙巳", "己":"甲午", "庚":"壬申", "辛":"癸酉", "壬":"丁亥", "癸":"丙子"}, # Hall of Words: Eloquence, teaching ability, and success in careers involving speech or writing.
}

pillar_shens = {
    '空亡': { # Emptiness/Void: A crucial modifier. It "hollows out" the pillar, making good stars less effective and bad stars less harmful.
        "甲子": "戌亥", "乙丑": "戌亥", "丙寅": "戌亥", "丁卯": "戌亥", "戊辰": "戌亥", "己巳": "戌亥", "庚午": "戌亥", "辛未": "戌亥", "壬申": "戌亥", "癸酉": "戌亥",
        "甲戌": "申酉", "乙亥": "申酉", "丙子": "申酉", "丁丑": "申酉", "戊寅": "申酉", "己卯": "申酉", "庚辰": "申酉", "辛巳": "申酉", "壬午": "申酉", "癸未": "申酉",
        "甲申": "午未", "乙酉": "午未", "丙戌": "午未", "丁亥": "午未", "戊子": "午未", "己丑": "午未", "庚寅": "午未", "辛卯": "午未", "壬辰": "午未", "癸巳": "午未",
        "甲午": "辰巳", "乙未": "辰巳", "丙申": "辰巳", "丁酉": "辰巳", "戊戌": "辰巳", "己亥": "辰巳", "庚子": "辰巳", "辛丑": "辰巳", "壬寅": "辰巳", "癸卯": "辰巳",
        "甲辰": "寅卯", "乙巳": "寅卯", "丙午": "寅卯", "丁未": "寅卯", "戊申": "寅卯", "己酉": "寅卯", "庚戌": "寅卯", "辛亥": "寅卯", "壬子": "寅卯", "癸丑": "寅卯",
        "甲寅": "子丑", "乙卯": "子丑", "丙辰": "子丑", "丁巳": "子丑", "戊午": "子丑", "己未": "子丑", "庚申": "子丑", "辛酉": "子丑", "壬戌": "子丑", "癸亥": "子丑"
    },
    '十恶大败': {"甲辰", "乙巳", "丙申", "丁亥", "戊戌", "己丑", "庚辰", "辛巳", "壬申", "癸亥"}, # Ten Evils Disaster: The 10 most inauspicious day pillars
}

# --- CORE ENGINE ---

def extract_shen_sha(lunar_birthday):
    """
    Extract all Shen Sha (神煞) stars from a BaZi chart using the provided lookup tables.
    Returns:
    - strs: [Year_Stars, Month_Stars, Day_Stars, Hour_Stars]
    - unique_shens: List of unique stars found
    - gans: Heavenly Stems [Y, M, D, H]
    - zhis: Earthly Branches [Y, M, D, H]
    """
    baZi = lunar_birthday.getEightChar()

    gans = [baZi.getYearGan(), baZi.getMonthGan(), baZi.getDayGan(), baZi.getTimeGan()]
    zhis = [baZi.getYearZhi(), baZi.getMonthZhi(), baZi.getDayZhi(), baZi.getTimeZhi()]

    me = gans[2]  # Day Stem (Day Master)
    day_pillar = gans[2] + zhis[2]  # Full day pillar

    strs = ['', '', '', '']  # Results per pillar
    all_found_shens = []

    # ============================================================
    # 1. YEAR BRANCH BASED (Year Branch → Other Pillars)
    # ============================================================
    for item, mapping in year_shens.items():
        lookup = mapping.get(zhis[0], "")
        for i in (1, 2, 3):  # Check Month, Day, Hour branches
            if zhis[i] in lookup:
                strs[i] = f"{strs[i]} {item}".strip()
                all_found_shens.append(item)

    # ============================================================
    # 2. MONTH BRANCH BASED (Month Branch → All Pillars)
    # ============================================================
    for item, mapping in month_shens.items():
        lookup = mapping.get(zhis[1], "")
        if item == '德秀':
            # Special: Check if any of the required stems are present anywhere in the gans
            for i in range(4):
                if gans[i] in lookup:
                    strs[i] = f"{strs[i]} {item}".strip()
                    all_found_shens.append(item)
        else:
            for i in range(4):
                # Checks both Stems (e.g. 天德) and Branches (e.g. 天医)
                if gans[i] in lookup or zhis[i] in lookup:
                    strs[i] = f"{strs[i]} {item}".strip()
                    all_found_shens.append(item)

    # ============================================================
    # 3. DAY BRANCH BASED (Day Branch → Other Pillars)
    # ============================================================
    for item, mapping in day_earthly_branches_shens.items():
        lookup = mapping.get(zhis[2], "")
        for i in (0, 1, 3):  # Check Year, Month, Hour branches
            if zhis[i] in lookup:
                strs[i] = f"{strs[i]} {item}".strip()
                all_found_shens.append(item)

    # ============================================================
    # 4. DAY STEM BASED (Day Stem → All Pillars)
    # ============================================================
    for item, mapping in day_heavenly_stem_shens.items():
        lookup = mapping.get(me, "")

        if item == '词馆':
            # Special case: 词馆 checks FULL pillar (stem + branch)
            for i in range(4):
                current_pillar = gans[i] + zhis[i]
                if current_pillar in lookup:
                    strs[i] = f"{strs[i]} {item}".strip()
                    all_found_shens.append(item)
        else:
            # Standard case: check only the branch
            for i in range(4):
                if zhis[i] in lookup:
                    strs[i] = f"{strs[i]} {item}".strip()
                    all_found_shens.append(item)

    # ============================================================
    # 5. PILLAR BASED (Day Pillar → All Branches)
    # ============================================================
    # Check for Kong Wang (Void)
    void_branches = pillar_shens['空亡'].get(day_pillar, "")
    for i in (0, 1, 3): # Check Year, Month, Hour branches
        if zhis[i] in void_branches:
            strs[i] = f"{strs[i]} 空亡".strip()
            all_found_shens.append("空亡")

    # Check for 十恶大败 (Shi E Da Bai) - Specifically on the Day Pillar
    if day_pillar in pillar_shens['十恶大败']:
        strs[2] = f"{strs[2]} 十恶大败".strip()
        all_found_shens.append("十恶大败")

    # ============================================================
    # 6. SPECIAL: Kui Gang (魁罡)
    # ============================================================
    kui_gang_pillars = ["庚辰", "庚戌", "戊戌", "壬辰"]
    if day_pillar in kui_gang_pillars:
        strs[2] = f"{strs[2]} 魁罡".strip()
        all_found_shens.append("魁罡")

    return strs, list(set(all_found_shens)), gans, zhis

from datetime import datetime as dt_class

def solar_to_datetime(solar_obj):
    """Convert a Solar object to a Python datetime object."""
    return dt_class(
        solar_obj.getYear(),
        solar_obj.getMonth(),
        solar_obj.getDay(),
        solar_obj.getHour(),
        solar_obj.getMinute(),
        solar_obj.getSecond()
    )

# --- EXECUTION ---

if __name__ == "__main__":
    # Desmond's birthday example
    solar_birthday= Solar.fromYmdHms(1985, 11, 25, 17, 7, 0)  # Create solar date
    solar_dt = solar_to_datetime(solar_birthday)
    tst_birthday, _ = get_true_solar_time(solar_dt, 1.4759, 103.808053)  # Get true solar time for the birthday

    print('阳历生日: ' + solar_birthday.toYmdHms())  # Print solar birthday
    print('真太阳时生日: ' + tst_birthday.toYmdHms())  # Print true solar time birthday

    # 节气表 Jiéqì Biǎo Solar terms (24 seasonal division points)
    lunar_birthday = tst_birthday.getLunar()  # Convert true solar time birthday to lunar calendar

    results, unique_shens, gans, zhis = extract_shen_sha(lunar_birthday)

    pillar_names = ["Year", "Month", "Day", "Hour"]
    print(f"Eight Char: {' '.join(gans)} / {' '.join(zhis)}\n")

    for i in range(4):
        print(f"{pillar_names[i]}: {gans[i]}{zhis[i]} | Stars: {results[i]}")

    print(f"\n--- Summary of All Stars Found ---")
    print(f"Total Unique Stars: {len(unique_shens)}")
    print(f"Stars: {', '.join(sorted(unique_shens))}")

阳历生日: 1985-11-25 17:07:00
真太阳时生日: 1985-11-25 16:14:28
Eight Char: 乙 丁 戊 庚 / 丑 亥 辰 申

Year: 乙丑 | Stars: 天德 德秀 天乙 国印 太极
Month: 丁亥 | Stars: 吊客 血刃 德秀 亡神 空亡
Day: 戊辰 | Stars: 红艳 太极
Hour: 庚申 | Stars: 天喜 大耗 元辰 文昌 福星 天厨

--- Summary of All Stars Found ---
Total Unique Stars: 16
Stars: 亡神, 元辰, 吊客, 国印, 大耗, 天乙, 天厨, 天喜, 天德, 太极, 德秀, 文昌, 福星, 空亡, 红艳, 血刃


In [51]:
# shens_infos = {
#     '孤辰': "孤僻、孤独：容易不合群、晚婚。男怕孤，女怕寡。",
#     '寡宿': "类似孤辰，主孤独。男怕孤，女怕寡。",
#     '大耗': "又名破碎，主意外破损、财帛损耗。",
#     '天德': "天德贵人：先天的福分，能化解凶煞，一生少险。",
#     '月德': "月德贵人：象征慈祥、智慧与祖荫，可逢凶化吉。",
#     '将星': "权柄之星，代表领导力、果断、有组织才干。",
#     '华盖': "艺术之星，主才华卓越但性情清高、孤僻，与宗教有缘。",
#     '驿马': "变动之星，主迁移、出差、奔波，利于事业扩张。",
#     '劫煞': "主争端、外伤、被骗。若为喜用则代表机智敏捷。",
#     '亡神': "主内心忧虑、是非纠纷。若为喜用则代表城府深、有谋略。",
#     '桃花': "咸池，主风流、美貌。日时见之为墙外桃花。",
#     '天乙': "天乙贵人：至尊之贵神，主逢凶化吉，多得贵人相助。",
#     '文昌': "文昌贵人：利于学业、考试、文笔，聪明灵秀。",
#     '阳刃': "极刚之星，主权力与暴烈。若无制约易生灾滞。",
#     '红艳': "红艳煞：主风流多情，气质浪漫，容易有情感纠葛。",
#     '金舆': "金舆贵人：主聪明、富贵，男得贤妻，女得良夫，生活安稳。",
#     '国印': "国印贵人：代表诚实、可靠、照章办事，易掌印信之权。",
#     '魁罡': "魁罡：性格刚毅、聪明，有领导权威，但人生起伏较大。",
#     '空亡': "空亡：象征虚无。吉星遇之福减，凶星遇之祸轻。"
# }